# Sea Ice Drift Skill assessment

## Import necessary libraries

In [1]:
# import necessary libraries
#! pip install cmocean
#! pip install geojson
#! pip install roaring_landmask
#! pip install coloredlogs
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from cmcrameri import cm
import cartopy as cy
import cartopy.crs as ccrs
import cartopy.feature as cf
import os
from glob import glob
from functools import partial
import pickle
from matplotlib.table import Table
import numpy.ma as ma
import opendrift
from datetime import datetime, timedelta
import logging
import opendrift
from opendrift.models.seaicedrift import SeaIceDrift
from opendrift.readers import reader_netCDF_CF_generic
from opendrift.readers import reader_global_landmask
import matplotlib.gridspec as gridspec
import numpy as np
from opendrift.models.physics_methods import skillscore_liu_weissberg
from opendrift.models.physics_methods import distance_between_trajectories, distance_along_trajectory
import matplotlib.ticker as mticker
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import partial
import pickle
from tqdm import tqdm
import pandas as pd


/home/maltem/.local/lib/python3.10/site-packages/pandas/core/arrays/masked.py:62: UserWarning: Pandas requires version '1.3.4' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


ModuleNotFoundError: No module named 'opendrift'

## Import buoy data

In [ ]:
# Load buoy data
with open('dr.pkl', 'rb') as f:
    dr = pickle.load(f)

## Initialize overarching dataset that saves all the relevant data for the different models

In [ ]:
# Make ONE dataset that saves all the relevant data for all the different models
model_names = ['NERSC-NextSIM', 'ECCC-RIOPS', 'ECCC-RIOPScoupled','NRL-GOFS', 'MET-TOPAZ5', 'MET-ROMS_m00', 'MET-ROMS_m01', 'MET-LAON', 'DMI-HYCOM']
start_dt = datetime(2024, 4, 1)
file_dates = [start_dt + timedelta(days=i) for i in range(0, 61)]
tr_nr_values = [x for x in range(1, 36) if x not in [5, 33]]
lt_int_values = [']'+str(i)+'_'+str(i+24)+']' for i in range(0,240,24)]
time_values = [start_dt + timedelta(minutes=30*i) for i in range(0,61*24*2+1)]  # 61 days * 24 hours per day * 2 (per half hour)


In [ ]:
# Create an empty DataArray with the required dimensions and NaN values
coords = {'model': model_names, 'lt_int': lt_int_values, 'tr_nr': tr_nr_values, 'time_ds': time_values}

# Utility function to create a DataArray with specified attributes
def create_dataarray(dims, shape, attrs):
    data = np.full(shape, np.nan, dtype=float)
    return xr.DataArray(data, coords={dim: coords[dim] for dim in dims}, dims=dims, attrs=attrs)

# Initialize dataset with specified variables for tr_nr and time only
data_vars = {
    'lat_obs': create_dataarray(['tr_nr', 'time_ds'], (len(tr_nr_values), len(time_values)), {'long_name': 'Observed latitude', 'units': 'degrees North'}),
    'lon_obs': create_dataarray(['tr_nr', 'time_ds'], (len(tr_nr_values), len(time_values)), {'long_name': 'Observed longitude', 'units': 'degrees East'}),
    'lat_sim': create_dataarray(['model', 'tr_nr', 'lt_int', 'time_ds'], (len(model_names), len(tr_nr_values), len(lt_int_values), len(time_values)), {'long_name': 'Simulated latitude', 'units': 'degrees North'}),
    'lon_sim': create_dataarray(['model', 'tr_nr', 'lt_int', 'time_ds'], (len(model_names), len(tr_nr_values), len(lt_int_values), len(time_values)), {'long_name': 'Simulated longitude', 'units': 'degrees East'}),
    'SIC': create_dataarray(['model', 'tr_nr', 'lt_int', 'time_ds'], (len(model_names), len(tr_nr_values), len(lt_int_values), len(time_values)), {'long_name': 'Sea ice concentration', 'units': '%'}),
    'Uice': create_dataarray(['model', 'tr_nr', 'lt_int', 'time_ds'], (len(model_names), len(tr_nr_values), len(lt_int_values), len(time_values)), {'long_name': 'Eastward sea ice velocity', 'units': 'm/s'}),
    'Vice': create_dataarray(['model', 'tr_nr', 'lt_int', 'time_ds'], (len(model_names), len(tr_nr_values), len(lt_int_values), len(time_values)), {'long_name': 'Northward sea ice velocity', 'units': 'm/s'}),
    'Hice': create_dataarray(['model', 'tr_nr', 'lt_int', 'time_ds'], (len(model_names), len(tr_nr_values), len(lt_int_values), len(time_values)), {'long_name': 'Sea ice thickness', 'units': 'm'}),
    'Sd': create_dataarray(['model', 'tr_nr', 'lt_int', 'time_ds'], (len(model_names), len(tr_nr_values), len(lt_int_values), len(time_values)), {'long_name': 'Snow depth', 'units': 'm'}),
    'AMSR2_SIC': create_dataarray(['tr_nr', 'time_ds'], (len(tr_nr_values), len(time_values)), {'long_name': 'AMSR2 sea ice concentration', 'units': '%'}),
    'MODIS_Ts': create_dataarray(['tr_nr', 'time_ds'], (len(tr_nr_values), len(time_values)), {'long_name': 'MODIS surface temperature', 'units': 'K'}),
}


# Create the dataset
dataset = xr.Dataset(data_vars)


In [ ]:
dataset

Fill in observed lon and lat positions of buoys

In [ ]:
for selected_tr_nr in tr_nr_values:
    
    # Nearest observed position (in time, to full half hour, if available)
    # Nearest-Neighbour interpolation with mask for closeness (<0.15h)
    lon_obs = dr[selected_tr_nr]['lon']
    lat_obs = dr[selected_tr_nr]['lat']
    time_pos_obs = dr[selected_tr_nr]['lon']['time']
    
    lon_obs_interp = lon_obs.interp(time=dataset.time_ds, method = 'nearest')
    lat_obs_interp = lat_obs.interp(time=dataset.time_ds, method = 'nearest')
    time_diff = np.abs((dataset.time_ds - time_pos_obs) / np.timedelta64(1, 'h'))
    min_time_diff = np.min(time_diff, axis=1)
    mask_obs_pos = (min_time_diff < 0.15).values
            
    lon_obs_interp = lon_obs_interp.where(mask_obs_pos, np.nan)
    lat_obs_interp = lat_obs_interp.where(mask_obs_pos, np.nan)
    
    dataset['lon_obs'].loc[dict(tr_nr=selected_tr_nr)] = lon_obs_interp
    dataset['lat_obs'].loc[dict(tr_nr=selected_tr_nr)] = lat_obs_interp
    

### Extra functions

Function to later rotate x, y sea ice velocities to u, v sea ice velocities

In [ ]:
# https://github.com/metno/NWPdocs/wiki/Examples#wind-direction-obtained-from-x-y-wind
# First, calculate local grid rotation (alpha) from input file
def distance(origin, destination):
    """
    (Source: https://stackoverflow.com/questions/19412462/
             getting-distance-between-two-points-based-on-latitude-longitude)

    Calculate the Haversine distance.

    Parameters
    ----------
    origin : tuple of float
        (lat, long)
    destination : tuple of float
        (lat, long)

    Returns
    -------
    distance_in_km : float

    Examples
    --------
    >>> origin = (48.1372, 11.5756)  # Munich
    >>> destination = (52.5186, 13.4083)  # Berlin
    >>> round(distance(origin, destination), 1)
    504.2
    """
    lat1, lon1 = origin
    lat2, lon2 = destination
    radius = 6371  # km

    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = (np.sin(dlat / 2) * np.sin(dlat / 2) +
         np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) *
         np.sin(dlon / 2) * np.sin(dlon / 2))
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    d = radius * c

    return d


Function to later determine map positions of buoy in model's map projection

In [ ]:
def interp_map_yx(example_model_ds, key_lon, key_lat, key_x, key_y, lon_obs, lat_obs, time_ds):
    yx_model = []

    # Loop through each position, find closest point (xc, yc) coordinates in model grid
    for time in time_ds:
        lon_obs_time = lon_obs.sel(time_ds=time)
        lat_obs_time = lat_obs.sel(time_ds=time)

        if np.isnan(lat_obs_time) or np.isnan(lon_obs_time):
            yx_model.append((np.nan, np.nan))
        else:
            # Determine the difference between grid lat/lon and the selected position
            c_model = np.maximum(np.abs(example_model_ds[key_lat] - lat_obs_time), np.abs(example_model_ds[key_lon] - lon_obs_time))
            # Determine the index of the point with minimal difference
            yi_model, xi_model = np.unravel_index(c_model.argmin(), c_model.shape)
            # Select corresponding xc, yc coordinates using dictionary indexing
            y_model_val = example_model_ds[key_y].isel({key_y: yi_model})
            x_model_val = example_model_ds[key_x].isel({key_x: xi_model})
            yx_model.append((y_model_val, x_model_val))
            #print(example_model_ds[key_lon].sel({key_y: y_model_val, key_x: x_model_val}).values, lon_obs_time.values)
            #print(example_model_ds[key_lat].sel({key_y: y_model_val, key_x: x_model_val}).values, lat_obs_time.values)

    # Convert the list of xc, yc values into separate y and x arrays
    y_model, x_model = zip(*yx_model)
    y_model = xr.DataArray(np.array(y_model), dims='time_ds', coords={'time_ds': time_ds})
    x_model = xr.DataArray(np.array(x_model), dims='time_ds', coords={'time_ds': time_ds})

    return y_model, x_model

def calculate_tr_nr_yx_model(example_model_ds, key_lon, key_lat, key_x, key_y, dataset, tr_nr):
    lon_obs = dataset.sel(tr_nr=tr_nr)['lon_obs']
    lat_obs = dataset.sel(tr_nr=tr_nr)['lat_obs']
    time_ds = dataset['time_ds']
    y_model, x_model = interp_map_yx(example_model_ds, key_lon, key_lat, key_x, key_y, lon_obs, lat_obs, time_ds)
    return tr_nr, y_model, x_model


# Sea ice drift models: concatenate forecasts and simulate drift

https://opendrift.github.io/tutorial.html

The most basic model, OceanDrift, is suitable for passive tracers (substances with no properties except for position, flowing with the ocean current) or objects at the ocean surface which may be subject to an additional wind drag

#1. Import a specific model for the relevant application

#2. Adding Readers

#3. Seeding elements

#4. Configuration

#5. Running the model

# Overview models

| **Model**                     | **Bulletin times (i)** | **Lead time**                                                                 | **First timestamp**      | **Simulation times**                                      | **Trajectory simulation times (30 min resolution)** | **File structure**           | **Velocity fields** | **Notes**                                                                                          |
|------------------------------|------------------------|--------------------------------------------------------------------------------|---------------------------|------------------------------------------------------------|-----------------------------------------------------|------------------------------|---------------------|-----------------------------------------------------------------------------------------------------|
| **DMI-HYCOM**                | Daily (00:00)          | 145h                                                                          | i + 1h                    | Hourly, 144 timesteps                                     | 287 timesteps from i + 1h                             | 1 file per simulation time step | X/Y                 | Available from 01/04-31/05. Missing files 27/04 and 28/04                                          |
| **ECCC-RIOPS**               | Daily (00:00)          | 85h                                                                           | i                         | Hourly, 85 timesteps                                      | 169 timesteps from i                                  | 1 file per simulation time step | X/Y                 | Available from 07/04-06/05                                                                         |
| **ECCC-RIOPScoupled**        | Daily (00:00)          | 49h                                                                           | i + 1h                    | Hourly, 48 timesteps                                     | 95 timesteps from i + 1h                              | 1 file per simulation time step | X/Y                 | Available from 01/04-06/05                                                                         |
| **MET-LAON**                 | Daily (00:00)          | 145h = 6d forecast                                                            | i + 1h                    | (roughly) hourly, 144 timesteps                           | 297 timesteps from i + 1h                             | 1 file per forecast            | X/Y                 | Available from 01/04-25/05                                                                         |
| **MET-ROMS (m00 and m01)**   | Daily (00:00)          | 97h                                                                           | i                         | Hourly, 97 timesteps                                      | 193 timesteps from i                                  | 1 file per forecast            | X/Y                 | Missing file 19/04. Available 01/04-31/05                                                           |
| **MET-TOPAZ5**               | Daily (00:00)          | 240h = 10d forecast<br><br>first day: +7d hindcast<br>fourth day: +1d hindcast | i - 0/1/7d + 30min         | Hourly, 240/264/408 timesteps                             | 479 timesteps from i + 30 min                         | 1 file per simulation day      | X/Y                 | Available 01/04-31/05                                                                             |
| **NERSC-NextSIM**            | Daily (00:00)          | 216h = 9d forecast<br>+ 24h = 1d hindcast                                      | i - 1d + 30min             | Hourly, 240 timesteps (tapering off towards end of May)   | 431 timesteps from i + 30min                          | 1 file per simulation day      | X/Y                 | Includes 1d hindcasts after 01/04, excluded for trajectory sims. Missing 20240413 for b20240405. Available 01/04-31/05 |
| **NRL-GOFS**                 | Daily (12:00)          | 180h = 7.5d forecast                                                           | i (12:00)                 | 12-hourly, 16 timesteps                                  | 361 timesteps from i (12:00)                          | 1 file per simulation time step | U/V                 | Outdated weather fields at bulletin date? Available 01/04-31/05                                    |


## NERSC-NextSIM

In [ ]:
model_name = 'NERSC-NextSIM'

In [ ]:
# NERSC - NextSIM-F
# Coordinates: y, x, longitude, latitude, time
# Variables: siconc (sea ice area fraction), sithick (sea ice thickness [m]), sisnthick (surface snow thickness [m]),
# vxsi (sea ice X velocity [m/s]), vysi (sea ice Y velocity [m/s]), sialb (sea ice albedo), siage (sea ice age [years]),
# siconc_my (sea ice area fraction of multi-year ice), siconc_young (sea ice area fraction of young ice)

# time = simulation time, roughly hourly from 00:30 - 23:30 for first date in file name
# initialization: b+date in file name
# simulation date: start of file name (24h)
# INCLUDES HINDCASTS
# MISSING FILE 20240413_hr-nersc-MODEL-nextsimf-ARC-b20240405-fv00.0.nc
example_nextsimf = xr.open_dataset(f"/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/NERSC-NextSIM/2024/04/20240403_hr-nersc-MODEL-nextsimf-ARC-b20240326-fv00.0.nc",engine="netcdf4")




In [ ]:
example_nextsimf.to_netcdf(f'example_{model_name}.nc')

In [ ]:
example_nextsimf = xr.open_dataset('example_NERSC-NextSIM.nc')
example_nextsimf

### Calculate local orientation of grid to rotate sea ice drift vectors

In [ ]:
# https://github.com/metno/NWPdocs/wiki/Examples#wind-direction-obtained-from-x-y-wind 
xx = example_nextsimf.x
yy = example_nextsimf.y
lat = example_nextsimf.latitude
lon = example_nextsimf.longitude
alpha=lat #Target matrix

for j in range(0,yy.size-1):
    for i in range(0,xx.size-1):
        # Prevent out of bounds
        if j==yy.size-1:
            j1=j-1; j2=j
        else:
            j1=j; j2=j+1
        if i==xx.size-1:
            i1=i-1; i2=i
        else:
            i1=i; i2=i+1

        dlatykm=distance([lat[j1,i1],lon[j1,i1]],[lat[j2,i1],lon[j1,i1]])
        dlonykm=distance([lat[j1,i1],lon[j1,i1]],[lat[j1,i1],lon[j2,i1]])
        
        alpha[j,i]=np.arctan2(dlatykm,dlonykm)*180/np.pi - 90
alpha_ds = xr.DataArray(alpha, dims=['y', 'x'])
alpha_ds.to_netcdf('alpha_'+model_name+'.nc')

### Create list of files of interest

In [ ]:
path_nextsimf = "/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/NERSC-NextSIM/2024/"
# some md5checksum and temp files as well => only select daily nc files
file_list_nextsimf_apr = list(glob(path_nextsimf + "04/*.nc"))
file_list_nextsimf_may = list(glob(path_nextsimf + "05/*.nc"))

file_list_nextsimf = file_list_nextsimf_apr + file_list_nextsimf_may
file_list_nextsimf.sort()


In [ ]:
file_list_nextsimf

### Determine relevant grid area + buffer zone

In [ ]:
used_area_nextsimf = np.zeros(example_nextsimf['latitude'].values.shape)

for tr_nr in [x for x in range(1, 36) if x not in [5, 33]]:    
    # loop through each position, indicate closest point in used_area_topaz5
    for time in dr[tr_nr]['time']:
        lat_time = dr[tr_nr]['lat'].sel(time=time).values
        lon_time = dr[tr_nr]['lon'].sel(time=time).values
        
        c_nextsimf = np.maximum(np.abs(example_nextsimf['latitude'].values-lat_time), np.abs(example_nextsimf['longitude'].values-lon_time))
        used_area_nextsimf[c_nextsimf == np.min(c_nextsimf)] = 1


In [ ]:
yi_nextsimf_min, xi_nextsimf_min = np.min(np.where(used_area_nextsimf == 1), axis=1)
yi_nextsimf_max, xi_nextsimf_max = np.max(np.where(used_area_nextsimf == 1), axis=1)
print(yi_nextsimf_min, xi_nextsimf_min)
print(yi_nextsimf_max, xi_nextsimf_max)

In [ ]:
yi_nextsimf_min, xi_nextsimf_min, yi_nextsimf_max, xi_nextsimf_max = 1121, 1471, 1294, 1558

In [ ]:
# add buffer for sea ice drift near edge
yi_nextsimf_min -= 50
xi_nextsimf_min -= 50
yi_nextsimf_max += 50
xi_nextsimf_max += 50
plt.imshow(example_nextsimf['siconc'].isel(time=0).isel(x=slice(xi_nextsimf_min,xi_nextsimf_max+1), y=slice(yi_nextsimf_min,yi_nextsimf_max+1)))


### Concatenate forecasts files to one file per initialisation date, only selecting relevant area and buffer zone
### Rename variables to reference names

In [ ]:
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)

start_dt = datetime(2024,4,1)
def _preprocess(x, x_bnds, y_bnds):
    ds = xr.Dataset()
    ds['SIC'] = x['siconc'].isel(x=slice(*x_bnds), y=slice(*y_bnds))
    ds['sea_ice_x_velocity'] = x['vxsi'].isel(x=slice(*x_bnds), y=slice(*y_bnds))
    ds['sea_ice_y_velocity'] = x['vysi'].isel(x=slice(*x_bnds), y=slice(*y_bnds))
    ds['Hice'] = x['sithick'].isel(x=slice(*x_bnds), y=slice(*y_bnds))
    ds['Sd'] = x['sisnthick'].isel(x=slice(*x_bnds), y=slice(*y_bnds))
    # redefine time to whole minutes
    ds['time'] = np.array(ds['time'] + np.timedelta64(1, 's'), dtype='datetime64[m]')
    return ds

partial_func = partial(_preprocess, \
                       x_bnds=(xi_nextsimf_min, xi_nextsimf_max+1), \
                       y_bnds=(yi_nextsimf_min, yi_nextsimf_max+1))

for date in [start_dt + timedelta(days=i) for i in range(0,61)]:
    date_str = date.strftime('%Y%m%d')
    print(date_str)
    file_list_nextsimf_date = list(glob(f"{path_nextsimf}*/*b{date_str}*.nc"))
    file_list_nextsimf_date.sort()
    nextsimf_ds_date=xr.open_mfdataset(file_list_nextsimf_date, combine='by_coords', preprocess=partial_func)
    nextsimf_ds_date.to_netcdf(os.path.join(output_folder, f"{model_name}_{date_str}.nc")) 
    

MISSING FILE FOR 2024/04/05: 20240413_hr-nersc-MODEL-nextsimf-ARC-b20240405-fv00.0.nc -> day less of data

In [ ]:
start_dt = datetime(2024,4,5)
output_folder = model_name + '_conc_forecasts'
a=xr.open_dataset(os.path.join(output_folder, f"{model_name}_{start_dt.strftime('%Y%m%d')}.nc"))
a.time.values

In [ ]:
date_str = datetime(2024,4,5).strftime('%Y%m%d')
a=list(glob(f"{path_nextsimf}*/*b{date_str}*.nc"))
a.sort()
a

In [ ]:
start_dt = datetime(2024,4,1)
for date in [start_dt + timedelta(days=i) for i in range(0,61)]:
    date_str = date.strftime('%Y%m%d')
    d=xr.open_dataset(os.path.join(output_folder, f"{model_name}_{date_str}.nc"))
    print(date)
    print(len(d.time.values))

### Simulate sea ice trajectories for each complete forecast

In [ ]:
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)
output_subfolder = os.path.join(output_folder, 'trajs_alt')
os.makedirs(output_subfolder, exist_ok=True) 
# forecast starts at 00:30
start_dt = datetime(2024, 4, 1, 0, 30)

reader_landmask = reader_global_landmask.Reader()

# Loop through dates
for file_date in [start_dt + timedelta(days=i) for i in range(0, 61)]:
    i = 0
    o = SeaIceDrift(loglevel=0)
    file_date_str = file_date.strftime('%Y%m%d')
    ds_nextsimf = xr.open_dataset(os.path.join(output_folder, f"{model_name}_{file_date_str}.nc"))
    # exclude hindcasts in the reader
    time_i_no_hind = np.where((ds_nextsimf.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h') > -0.1)[0]
    ds_nextsimf = ds_nextsimf.isel(time = time_i_no_hind)
    reader_nextsimf = reader_netCDF_CF_generic.Reader(
        ds_nextsimf,
        proj4='+proj=stere +lat_0=90 +lat_ts=90 +lon_0=-45 +x_0=0 +y_0=0 +R=6378273 +ellps=sphere +units=m +no_defs'
    )

    o.add_reader([reader_landmask, reader_nextsimf])
    for tr_nr in [x for x in range(1, 36) if x not in [5, 33]]:
        lon0 = dr[tr_nr]['lon'].interp(time=file_date, method = 'nearest')
        lat0 = dr[tr_nr]['lat'].interp(time=file_date, method = 'nearest')
        time_diff = np.abs((dr[tr_nr]['lon']['time']- np.datetime64(file_date)) / np.timedelta64(1, 'h'))
        min_time_diff = np.min(time_diff).values

        if (min_time_diff<0.15) and not (np.isnan(lon0) or np.isnan(lat0)):
            o.seed_elements(lon=lon0, lat=lat0, time=file_date, origin_marker=tr_nr)
            print(f"{tr_nr} seeded")
            
            i = 1
            
    if i != 0:
    # run with 30 minute resolution up to end time of the forecast
        o.run(end_time=reader_nextsimf.end_time, time_step=1800, time_step_output=1800, outfile=os.path.join(output_subfolder, f"trajs_{model_name}_{file_date_str}.nc"))


Plot trajectory simulation to compare later with reconstruction

In [ ]:
start_dt = datetime(2024,4,16)
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)
output_subfolder = os.path.join(output_folder, 'trajs_alt')
os.makedirs(output_subfolder, exist_ok=True) 
file_date_str = start_dt.strftime('%Y%m%d')
a=xr.open_dataset(os.path.join(output_subfolder, f"trajs_{model_name}_{file_date_str}.nc"))
a['lon'].sel(trajectory=3).plot()

### Fill in overarching datast

In [ ]:
# Initialize variables and paths
model_name = 'NERSC-NextSIM'
example_model_ds = xr.open_dataset(f'example_{model_name}.nc',engine="netcdf4")
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)
output_subfolder = os.path.join(output_folder, 'trajs_alt')
os.makedirs(output_subfolder, exist_ok=True)

key_lon, key_lat, key_x, key_y = 'longitude', 'latitude', 'x', 'y'
alpha_ds = xr.open_dataset('alpha_' + model_name + '.nc')

In [ ]:
# Calculate yx_model in parallel
yx_model = {}
with ThreadPoolExecutor() as executor:
    futures = {executor.submit(calculate_tr_nr_yx_model, example_model_ds, key_lon, key_lat, key_x, key_y, dataset, tr_nr): tr_nr for tr_nr in tr_nr_values}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Calculating yx_model"):
        tr_nr, y_model, x_model = future.result()
        yx_model[tr_nr] = (y_model, x_model)
        
with open(f'yx_model_{model_name}.pkl', 'wb') as f:
    pickle.dump(yx_model,f)

In [ ]:
with open(f'yx_model_{model_name}.pkl', 'rb') as f:
    yx_model = pickle.load(f)

### Fill in overarching dataset with forecast variables

In [ ]:
# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

def process_forecasts(file_date, output_folder, model_name, alpha_ds, key_lon, key_lat, dataset, yx_model):
    file_date_str = file_date.strftime('%Y%m%d')
    logging.info(f"Processing forecasts for date: {file_date_str}")

    # Try opening forecasts
    file_name_fc = os.path.join(output_folder, f"{model_name}_{file_date_str}.nc")
    try:
        ds_fc = xr.open_dataset(file_name_fc)   
        alpha_rad = alpha_ds['__xarray_dataarray_variable__'] * np.pi / 180
        ds_fc['Uice'] = ds_fc['sea_ice_x_velocity'] * np.cos(alpha_rad) - ds_fc['sea_ice_y_velocity'] * np.sin(alpha_rad)
        ds_fc['Vice'] = ds_fc['sea_ice_x_velocity'] * np.sin(alpha_rad) + ds_fc['sea_ice_y_velocity'] * np.cos(alpha_rad)

        time_ds = dataset.time_ds
        updated_dataset = dataset.copy()

        for tr_nr, (y_model, x_model) in yx_model.items():
            mask_nan = ~(np.isnan(x_model) + np.isnan(y_model))
            for i in range(0, 240, 24):
                lt_int = f']{i}_{i+24}]'
                left_lt = ((ds_fc.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))> i+0.1
                right_lt = ((ds_fc.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))<= i+24+0.1
                sel_lt = np.where(left_lt*right_lt)[0]
                ds_fc_lt_int = ds_fc.isel(time=sel_lt)
                av_times_fc = ds_fc_lt_int['time'].values
                mask_av = pd.to_datetime(time_ds).isin(pd.to_datetime(av_times_fc))

                if np.sum(mask_av * mask_nan) > 0:
                    for var in ds_fc.data_vars:
                        if var not in ['sea_ice_x_velocity', 'sea_ice_y_velocity']:
                            #print(x_model[mask_av * mask_nan])
                            var_model_lt_int_path = ds_fc[var].sel({'time': time_ds[mask_av * mask_nan], key_x: x_model[mask_av * mask_nan], key_y: y_model[mask_av * mask_nan]})
                            updated_dataset[var].loc[dict(model=model_name, tr_nr=tr_nr, lt_int=lt_int, time_ds=time_ds[mask_av * mask_nan])] = var_model_lt_int_path.values


        return updated_dataset
        
    except FileNotFoundError:
        logging.warning(f"File not found: {file_name_fc}")
        return dataset



# Process forecasts in parallel
with ThreadPoolExecutor() as executor:
    futures = [
        executor.submit(partial(process_forecasts, file_date, output_folder, model_name, alpha_ds, key_lon, key_lat, dataset, yx_model))
        for file_date in tqdm(file_dates, desc="Processing forecasts")
    ]
    for future in tqdm(futures, desc="Updating dataset with forecast results"):
        result = future.result()
        if result is not None:
            dataset.update(result)

### Fill in overarching dataset with simulated drift trajectories

In [ ]:
def process_trajectories(file_date, output_subfolder, model_name, dataset):
    file_date_str = file_date.strftime('%Y%m%d')
    logging.info(f"Processing trajectories for date: {file_date_str}")

    file_name_traj = os.path.join(output_subfolder, f"trajs_{model_name}_{file_date_str}.nc")
    try:
        ds_traj = xr.open_dataset(file_name_traj)
        updated_dataset = dataset.copy()
        trajectory_sim = 0

        for tr_nr in np.unique(ds_traj['origin_marker'].values):
            if not np.isnan(tr_nr):
                trajectory_sim += 1
                for i in range(0, 240, 24):
                    lt_int = f']{i}_{i + 24}]'
                    left_lt = ((ds_traj.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))> i+0.1
                    right_lt = ((ds_traj.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))<= i+24+0.1
                    sel_lt = np.where(left_lt*right_lt)[0]                
                    time_traj = ds_traj.sel(trajectory=trajectory_sim)['time'].isel(time=sel_lt)
                    time_traj = time_traj.sel(time = (time_traj.time <= np.datetime64('2024-06-01T00:00')))
                    if len(time_traj.values)>0:
                        updated_dataset['lon_sim'].loc[dict(model=model_name, tr_nr=tr_nr, lt_int=lt_int, time_ds=time_traj)] = ds_traj['lon'].sel(trajectory=trajectory_sim, time=time_traj)
                        updated_dataset['lat_sim'].loc[dict(model=model_name, tr_nr=tr_nr, lt_int=lt_int, time_ds=time_traj)] = ds_traj['lat'].sel(trajectory=trajectory_sim, time=time_traj)

        return updated_dataset

    except FileNotFoundError:
        logging.warning(f"File not found: {file_name_traj}")
        return dataset

# Process trajectories in parallel
with ThreadPoolExecutor() as executor:
    futures = [
        executor.submit(partial(process_trajectories, file_date, output_subfolder, model_name, dataset))
        for file_date in tqdm(file_dates, desc="Processing trajectories")
    ]
    for future in tqdm(futures, desc="Updating dataset with trajectory results"):
        result = future.result()
        if result is not None:
            dataset.update(result)


### Plot to check data

In [ ]:
buoy_nr = 3
buoy_data = dataset.sel(tr_nr = buoy_nr)

colors = {
    'NERSC-NextSIM': 'tab:blue',
    'ECCC-RIOPS': 'tab:orange',
    'ECCC-RIOPScoupled': 'tab:green',
    'NRL-GOFS': 'tab:red',
    'MET-TOPAZ5': 'tab:purple',
    'MET-ROMS_m00': 'tab:cyan',
    'MET-ROMS_m01': 'tab:pink'
}

#['NERSC-NextSIM', 'ECCC-RIOPS', 'ECCC-RIOPScoupled', 'NRL-GOFS','MET-TOPAZ5', 'MET-ROMS_m00', 'MET-ROMS_m01']
#models_sel = []
models_sel = ['NERSC-NextSIM']
target_date = '2024-04-16'  # Example date in YYYY-MM-DD format

lon_obs_b, lat_obs_b = buoy_data['lon_obs'], buoy_data['lat_obs']
plt.figure()
start = pd.to_datetime(target_date) + pd.DateOffset(seconds=1)
end = start + pd.DateOffset(days=10)
        
lon_obs_b.sel(time_ds=slice(start,end)).plot(color='k')
for model_name in models_sel:
    color = colors.get(model_name, 'black')
    
    # Plot dummy line for legend
    plt.plot([], [], color=color, label=model_name)
    
    for k in range(0,10):
        lt_int_sel = dataset['lt_int'].values[k]
        start = pd.to_datetime(target_date) + pd.DateOffset(seconds=1) + pd.DateOffset(days=k) 
        end = start + pd.DateOffset(days=1)
        lon_sim_lt_k = buoy_data['lon_sim'].sel(model=model_name, time_ds=slice(start,end), lt_int=lt_int_sel)
        lat_sim_lt_k = buoy_data['lat_sim'].sel(model=model_name, time_ds=slice(start,end), lt_int=lt_int_sel)
        lon_sim_lt_k.plot(color=color)

plt.legend(loc='upper left')
plt.grid(True)

In [ ]:
model_name = 'NERSC-NextSIM'
dataset['lat_obs'].sel(tr_nr=4).plot()
dataset['lat_sim'].sel(tr_nr=4, model=model_name, lt_int=']0_24]').dropna(dim='time_ds', how='any').plot()

In [ ]:
dataset['SIC'].sel(tr_nr=1, model=model_name, lt_int=']0_24]').dropna(dim='time_ds', how='any').plot()

In [ ]:
dataset.to_netcdf('traj_ds_nextsimf.nc')

## MET-TOPAZ5

In [ ]:
model_name = 'MET-TOPAZ5'

In [ ]:
# MET TOPAZ-5
# Coordinates: time, y, x
# Variables: model_depth (sea_floor_depth_below_sea_level), so (sea_water_salinity [1e-3]),
# thetao (sea_water_potential_temperature [degrees Celcius]), vxo (sea_water_x_velocity [m/s]),
# vyo (sea_water_y_velocity [m/s]), zos (sea_surface_height_above_geoid [m]), sithick (sea_ice_thickness [m]),
# siconc (sea_ice_area_fraction), vxsi (sea_ice_x_velocity [m/s]), vysi (sea_ice_y_velocity [m/s]),
# sisnthick (surface_snow_thickness [m])

# time = simulation time, hourly from 00:30 - 23:30
# initialization: b+date in file name
# INCLUDES HINDCASTS (sometimes)
example_topaz5 = xr.open_dataset(f"/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/MET-TOPAZ5/metnotopaz5/arctic_1hr/mersea-class1/2024/05/20240531_hr-metno-MODEL-topaz5-ARC-b20240603-fv02.0.nc",engine="netcdf4")


In [ ]:
example_topaz5.to_netcdf(f'example_{model_name}.nc')

In [ ]:
example_topaz5 = xr.open_dataset('example_MET-TOPAZ5.nc')
example_topaz5

### Calculate local orientation of grid to rotate sea ice drift vectors

In [ ]:
# Variables 
xx = example_topaz5.x
yy = example_topaz5.y
lat = example_topaz5.latitude
lon = example_topaz5.longitude
alphaa=lat #Target matrix

for j in range(0,yy.size-1):
    for i in range(0,xx.size-1):
        # Prevent out of bounds
        if j==yy.size-1:
            j1=j-1; j2=j
        else:
            j1=j; j2=j+1
        if i==xx.size-1:
            i1=i-1; i2=i
        else:
            i1=i; i2=i+1

        dlatykm=distance([lat[j1,i1],lon[j1,i1]],[lat[j2,i1],lon[j1,i1]])
        dlonykm=distance([lat[j1,i1],lon[j1,i1]],[lat[j1,i1],lon[j2,i1]])
        
        alphaa[j,i]=np.arctan2(dlatykm,dlonykm)*180/np.pi - 90
alphaa_ds = xr.DataArray(alphaa, dims=['y', 'x'])
alphaa_ds.to_netcdf('alpha_'+model_name+'.nc')

### Create list of files of interest

In [ ]:
path_topaz5 = "/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/MET-TOPAZ5/metnotopaz5/arctic_1hr/mersea-class1/2024/"
# some md5checksum and temp files as well => only select daily nc files
file_list_topaz5_apr = list(glob(path_topaz5 + "04/*.nc"))
file_list_topaz5_may = list(glob(path_topaz5 + "05/*.nc"))

file_list_topaz5 = file_list_topaz5_apr + file_list_topaz5_may
file_list_topaz5.sort()


In [ ]:
file_list_topaz5

### Determine relevant grid area + buffer zone

In [ ]:
used_area_topaz5 = np.zeros(example_topaz5['latitude'].values.shape)

for tr_nr in [x for x in range(1, 36) if x not in [5, 33]]:
    
    # loop through each position, indicate closest point in used_area_topaz5
    for time in dr[tr_nr]['time']:
        lat_time = dr[tr_nr]['lat'].sel(time=time).values
        lon_time = dr[tr_nr]['lon'].sel(time=time).values
        
        c_topaz5 = np.maximum(np.abs(example_topaz5['latitude'].values-lat_time), np.abs(example_topaz5['longitude'].values-lon_time))
        used_area_topaz5[c_topaz5 == np.min(c_topaz5)] = 1


In [ ]:
yi_topaz5_min, xi_topaz5_min = np.min(np.where(used_area_topaz5 == 1), axis=1)
yi_topaz5_max, xi_topaz5_max = np.max(np.where(used_area_topaz5 == 1), axis=1)
print(yi_topaz5_min, xi_topaz5_min)
print(yi_topaz5_max, xi_topaz5_max)

In [ ]:
yi_topaz5_min, xi_topaz5_min, yi_topaz5_max, xi_topaz5_max = 538, 706, 621, 747

In [ ]:
# add buffer for sea ice drift near edge
yi_topaz5_min -= 50
xi_topaz5_min -= 50
yi_topaz5_max += 50
xi_topaz5_max += 50
plt.imshow(example_topaz5['siconc'].isel(time=0).isel(x=slice(xi_topaz5_min,xi_topaz5_max+1), y=slice(yi_topaz5_min,yi_topaz5_max+1)))


### Concatenate forecasts files to one file per initialisation date, only selecting relevant area and buffer zone
### Rename variables to reference names

In [ ]:
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)

start_dt = datetime(2024,4,1)
def _preprocess(x, x_bnds, y_bnds):
    ds = xr.Dataset()
    ds['SIC'] = x['siconc'].isel(x=slice(*x_bnds), y=slice(*y_bnds))
    ds['sea_ice_x_velocity'] = x['vxsi'].isel(x=slice(*x_bnds), y=slice(*y_bnds))
    ds['sea_ice_y_velocity'] = x['vysi'].isel(x=slice(*x_bnds), y=slice(*y_bnds))
    ds['Hice'] = x['sithick'].isel(x=slice(*x_bnds), y=slice(*y_bnds))
    ds['Sd'] = x['sisnthick'].isel(x=slice(*x_bnds), y=slice(*y_bnds))
    return ds
partial_func = partial(_preprocess, \
                       x_bnds=(xi_topaz5_min, xi_topaz5_max+1), \
                       y_bnds=(yi_topaz5_min, yi_topaz5_max+1))

for date in [start_dt + timedelta(days=i) for i in range(50,61)]:
    date_str = date.strftime('%Y%m%d')
    print(date_str)
    file_list_topaz5_date = list(glob(f"{path_topaz5}*/*b{date_str}*.nc"))
    file_list_topaz5_date.sort()
    topaz5_ds_date=xr.open_mfdataset(file_list_topaz5_date, combine='by_coords', preprocess=partial_func)
    topaz5_ds_date.to_netcdf(os.path.join(output_folder, f"{model_name}_{date_str}.nc")) 
    

10d forecast

first day of week: +7d hindcast

third day of week: +1d hindcast

In [ ]:
start_dt = datetime(2024,4,1)
for date in [start_dt + timedelta(days=i) for i in range(0,61)]:
    path_topaz5 = "/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/MET-TOPAZ5/metnotopaz5/arctic_1hr/mersea-class1/2024/"
    date_str = date.strftime('%Y%m%d')
    a=list(glob(f"{path_topaz5}*/*b{date_str}*.nc"))
    print(date_str)
    print(len(a))

In [ ]:
start_dt = datetime(2024,4,4)
output_folder = model_name + '_conc_forecasts'
a=xr.open_dataset(os.path.join(output_folder, f"{model_name}_{start_dt.strftime('%Y%m%d')}.nc"))
print(a.time.values)
print(len(a.time.values))

### Simulate sea ice trajectories for each complete forecast

In [ ]:
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)
output_subfolder = os.path.join(output_folder, 'trajs_alt')
os.makedirs(output_subfolder, exist_ok=True) 
# forecast (not incl hindcasts) starts at 00:00
start_dt = datetime(2024,4,1, 0, 30)
reader_landmask = reader_global_landmask.Reader()

# Loop through dates
for file_date in [start_dt + timedelta(days=i) for i in range(0, 61)]:
    i = 0
    o = SeaIceDrift(loglevel=0)
    file_date_str = file_date.strftime('%Y%m%d')
    ds_topaz5 = xr.open_dataset(os.path.join(output_folder, f"{model_name}_{file_date_str}.nc"))
    
    # exclude hindcasts in the reader
    time_i_no_hind = np.where((ds_topaz5.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h') > -0.1)[0]
    ds_topaz5 = ds_topaz5.isel(time = time_i_no_hind)
    
    reader_topaz5 = reader_netCDF_CF_generic.Reader(
        ds_topaz5,
        proj4='+proj=stere +lon_0=-45 +lat_0=90 +k=1 +R=6378273 +no_defs'
    )

    o.add_reader([reader_landmask, reader_topaz5])
    for tr_nr in [x for x in range(1, 36) if x not in [5, 33]]:
    
        lon0 = dr[tr_nr]['lon'].interp(time=file_date, method = 'nearest')
        lat0 = dr[tr_nr]['lat'].interp(time=file_date, method = 'nearest')
        time_diff = np.abs((dr[tr_nr]['lon']['time']- np.datetime64(file_date)) / np.timedelta64(1, 'h'))
        min_time_diff = np.min(time_diff).values

        if (min_time_diff<0.15) and not (np.isnan(lon0) or np.isnan(lat0)):

            o.seed_elements(lon=lon0, lat=lat0, time=file_date, origin_marker=tr_nr)
            print(f"{tr_nr} seeded")
            
            i = 1
            
    if i != 0:
        # run with 30 minute resolution up to end time of the forecast
        o.run(end_time=reader_topaz5.end_time, time_step=1800, time_step_output=1800, outfile=os.path.join(output_subfolder, f"trajs_{model_name}_{file_date_str}.nc"))


Plot trajectory simulation to compare later with reconstruction

In [ ]:
start_dt = datetime(2024,4,16)
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)
output_subfolder = os.path.join(output_folder, 'trajs_alt')
os.makedirs(output_subfolder, exist_ok=True) 
file_date_str = start_dt.strftime('%Y%m%d')
a=xr.open_dataset(os.path.join(output_subfolder, f"trajs_{model_name}_{file_date_str}.nc"))
a
a['lon'].sel(trajectory=3).plot()

### Fill in overarching dataset

In [ ]:
# Initialize variables and paths
model_name = 'MET-TOPAZ5'
example_model_ds = xr.open_dataset(f'example_{model_name}.nc',engine="netcdf4")
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)
output_subfolder = os.path.join(output_folder, 'trajs_alt')
os.makedirs(output_subfolder, exist_ok=True)

key_lon, key_lat, key_x, key_y = 'longitude', 'latitude', 'x', 'y'
alpha_ds = xr.open_dataset('alpha_' + model_name + '.nc')

In [ ]:
# Calculate yx_model in parallel
yx_model = {}
with ThreadPoolExecutor() as executor:
    futures = {executor.submit(calculate_tr_nr_yx_model, example_model_ds, key_lon, key_lat, key_x, key_y, dataset, tr_nr): tr_nr for tr_nr in tr_nr_values}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Calculating yx_model"):
        tr_nr, y_model, x_model = future.result()
        yx_model[tr_nr] = (y_model, x_model)
        
with open(f'yx_model_{model_name}.pkl', 'wb') as f:
    pickle.dump(yx_model,f)

In [ ]:
with open(f'yx_model_{model_name}.pkl', 'rb') as f:
    yx_model = pickle.load(f)

### Fill in overarching dataset with forecast variables

In [ ]:
# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

def process_forecasts(file_date, output_folder, model_name, alpha_ds, key_lon, key_lat, dataset, yx_model):
    file_date_str = file_date.strftime('%Y%m%d')
    logging.info(f"Processing forecasts for date: {file_date_str}")

    # Try opening forecasts
    file_name_fc = os.path.join(output_folder, f"{model_name}_{file_date_str}.nc")
    try:
        ds_fc = xr.open_dataset(file_name_fc)
        
        alpha_rad = alpha_ds['__xarray_dataarray_variable__'] * np.pi / 180
        ds_fc['Uice'] = ds_fc['sea_ice_x_velocity'] * np.cos(alpha_rad) - ds_fc['sea_ice_y_velocity'] * np.sin(alpha_rad)
        ds_fc['Vice'] = ds_fc['sea_ice_x_velocity'] * np.sin(alpha_rad) + ds_fc['sea_ice_y_velocity'] * np.cos(alpha_rad)

        time_ds = dataset.time_ds
        updated_dataset = dataset.copy()

        for tr_nr, (y_model, x_model) in yx_model.items():
            mask_nan = ~(np.isnan(x_model) + np.isnan(y_model))
            for i in range(0, 240, 24):
                lt_int = f']{i}_{i+24}]'
                left_lt = ((ds_fc.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))> i+0.1
                right_lt = ((ds_fc.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))<= i+24+0.1
                sel_lt = np.where(left_lt*right_lt)[0]
                ds_fc_lt_int = ds_fc.isel(time=sel_lt)
                av_times_fc = ds_fc_lt_int['time'].values
                mask_av = pd.to_datetime(time_ds).isin(pd.to_datetime(av_times_fc))

                if np.sum(mask_av * mask_nan) > 0:
                    for var in ds_fc.data_vars:
                        if var not in ['sea_ice_x_velocity', 'sea_ice_y_velocity']:
                            var_model_lt_int_path = ds_fc[var].sel({'time': time_ds[mask_av * mask_nan], key_x: x_model[mask_av * mask_nan], key_y: y_model[mask_av * mask_nan]})
                            updated_dataset[var].loc[dict(model=model_name, tr_nr=tr_nr, lt_int=lt_int, time_ds=time_ds[mask_av * mask_nan])] = var_model_lt_int_path.values
        return updated_dataset
    except FileNotFoundError:
        logging.warning(f"File not found: {file_name_fc}")
        return dataset



# Process forecasts in parallel
with ThreadPoolExecutor() as executor:
    futures = [
        executor.submit(partial(process_forecasts, file_date, output_folder, model_name, alpha_ds, key_lon, key_lat, dataset, yx_model))
        for file_date in tqdm(file_dates, desc="Processing forecasts")
    ]
    for future in tqdm(futures, desc="Updating dataset with forecast results"):
        result = future.result()
        if result is not None:
            dataset.update(result)

### Fill in overarching dataset with simulated drift trajectories

In [ ]:
def process_trajectories(file_date, output_subfolder, model_name, dataset):
    file_date_str = file_date.strftime('%Y%m%d')
    logging.info(f"Processing trajectories for date: {file_date_str}")

    file_name_traj = os.path.join(output_subfolder, f"trajs_{model_name}_{file_date_str}.nc")
    try:
        ds_traj = xr.open_dataset(file_name_traj)
        updated_dataset = dataset.copy()
        trajectory_sim = 0

        for tr_nr in np.unique(ds_traj['origin_marker'].values):
            if not np.isnan(tr_nr):
                trajectory_sim += 1
                for i in range(0, 240, 24):
                    lt_int = f']{i}_{i + 24}]'
                    left_lt = ((ds_traj.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))> i+0.1
                    right_lt = ((ds_traj.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))<= i+24+0.1
                    sel_lt = np.where(left_lt*right_lt)[0]                
                    time_traj = ds_traj.sel(trajectory=trajectory_sim)['time'].isel(time=sel_lt)
                    time_traj = time_traj.sel(time = (time_traj.time <= np.datetime64('2024-06-01T00:00')))
                    
                    if len(time_traj.values)>0:
                        updated_dataset['lon_sim'].loc[dict(model=model_name, tr_nr=tr_nr, lt_int=lt_int, time_ds=time_traj)] = ds_traj['lon'].sel(trajectory=trajectory_sim, time=time_traj)
                        updated_dataset['lat_sim'].loc[dict(model=model_name, tr_nr=tr_nr, lt_int=lt_int, time_ds=time_traj)] = ds_traj['lat'].sel(trajectory=trajectory_sim, time=time_traj)

        return updated_dataset
    
    except FileNotFoundError:
        logging.warning(f"File not found: {file_name_traj}")
        return dataset
    
# Process trajectories in parallel
with ThreadPoolExecutor() as executor:
    futures = [
        executor.submit(partial(process_trajectories, file_date, output_subfolder, model_name, dataset))
        for file_date in tqdm(file_dates, desc="Processing trajectories")
    ]
    for future in tqdm(futures, desc="Updating dataset with trajectory results"):
        result = future.result()
        if result is not None:
            dataset.update(result)


### Plot to check data

In [ ]:
buoy_nr = 3
buoy_data = dataset.sel(tr_nr = buoy_nr)

colors = {
    'NERSC-NextSIM': 'tab:blue',
    'ECCC-RIOPS': 'tab:orange',
    'ECCC-RIOPScoupled': 'tab:green',
    'NRL-GOFS': 'tab:red',
    'MET-TOPAZ5': 'tab:purple',
    'MET-ROMS_m00': 'tab:cyan',
    'MET-ROMS_m01': 'tab:pink'
}

#['NERSC-NextSIM', 'ECCC-RIOPS', 'ECCC-RIOPScoupled', 'NRL-GOFS','MET-TOPAZ5', 'MET-ROMS_m00', 'MET-ROMS_m01']
#models_sel = []
models_sel = ['MET-TOPAZ5']
target_date = '2024-04-16'  # Example date in YYYY-MM-DD format

lon_obs_b, lat_obs_b = buoy_data['lon_obs'], buoy_data['lat_obs']
plt.figure()
start = pd.to_datetime(target_date) + pd.DateOffset(seconds=1)
end = start + pd.DateOffset(days=9)
        
lon_obs_b.sel(time_ds=slice(start,end)).plot(color='k')
for model_name in models_sel:
    color = colors.get(model_name, 'black')
    
    # Plot dummy line for legend
    plt.plot([], [], color=color, label=model_name)
    
    for k in range(0,9):
        lt_int_sel = dataset['lt_int'].values[k]
        start = pd.to_datetime(target_date) + pd.DateOffset(seconds=1) + pd.DateOffset(days=k) 
        end = start + pd.DateOffset(days=1)
        lon_sim_lt_k = buoy_data['lon_sim'].sel(model=model_name, time_ds=slice(start,end), lt_int=lt_int_sel)
        lat_sim_lt_k = buoy_data['lat_sim'].sel(model=model_name, time_ds=slice(start,end), lt_int=lt_int_sel)
        lon_sim_lt_k.plot(color=color)

plt.legend(loc='upper left')
plt.grid(True)

In [ ]:
model_name = 'MET-TOPAZ5'
dataset['lat_obs'].sel(tr_nr=4).plot()
dataset['lat_sim'].sel(tr_nr=4, model=model_name, lt_int=']0_24]').dropna(dim='time_ds', how='any').plot()
dataset['lat_sim'].sel(tr_nr=4, model='NERSC-NextSIM', lt_int=']0_24]').dropna(dim='time_ds', how='any').plot()

In [ ]:
dataset['SIC'].sel(tr_nr=1, model=model_name, lt_int=']0_24]').dropna(dim='time_ds', how='any').plot()

In [ ]:
dataset.to_netcdf('traj_ds_TOPAZ5.nc')

## NRL-GOFS

In [ ]:
model_name = 'NRL-GOFS'

In [ ]:
# NRL-GOFS
# Coordinates: time, lat, lon
# Variables: Tair (air_temperature [K]), Tsfc (surface_temperature_where_sea_ice [K]), aice (sea_ice_area_fraction)
# hi (sea_ice_thickness [m]), sst (sea_water_temperature [degC]), uvel (eastward_sea_ice_velocity [m/s]), 
# vvel (northward_sea_ice_velocity)

# time = simulation time, 1 per file, txxx: xxx hours after initialization
# initialization in file name / bulletin date (always at 12)

example_gofs = xr.open_dataset(f'/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/NRL-GOFS/hycom-cice_inst_ARCu0.08_930_2024053112_t000_SvalMIZ.nc')


In [ ]:
example_gofs.to_netcdf(f'example_{model_name}.nc')

In [ ]:
example_gofs = xr.open_dataset('example_NRL-GOFS.nc')

In [ ]:
example_gofs['uvel'].isel(time=0).plot()

### No need to calculate local grid orientation, cause given in U/V

In lat/lon grid: X/Y direction correspond with U/V

### Create list of files of interest

In [ ]:
path_gofs = "/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/NRL-GOFS/"
file_list_gofs = list(glob(path_gofs + "*SvalMIZ.nc"))

file_list_gofs.sort()
print(file_list_gofs)

### Determine relevant grid area + buffer zone

In [ ]:
min_lon, min_lat, max_lon, max_lat = 400, 400, -400, -400

for tr_nr in [x for x in range(1, 36) if x not in [5, 33]]:
    
    # loop through each position, indicate closest point in used_area_topaz5
    for time in dr[tr_nr]['time']:
        lat_time = dr[tr_nr]['lat'].sel(time=time)
        lon_time = dr[tr_nr]['lon'].sel(time=time)
        if lon_time < min_lon:
            min_lon = lon_time
        if lat_time < min_lat:
            min_lat = lat_time
        if lon_time > max_lon:
            max_lon = lon_time
        if lat_time > max_lat:
            max_lat = lat_time

In [ ]:
loni_gofs_slice_start = 224
loni_gofs_slice_end = 510
lati_gofs_slice_start = 212
lati_gofs_slice_end = 287

In [ ]:
print(min_lon.values, max_lon.values)
loni_gofs_slice_start = 224
loni_gofs_slice_end = 510
lon_slice_ex_gofs = example_gofs.lon.isel(lon=slice(loni_gofs_slice_start,loni_gofs_slice_end)).values
print(lon_slice_ex_gofs[0], lon_slice_ex_gofs[-1])

In [ ]:
print(min_lat.values, max_lat.values)
lati_gofs_slice_start = 212
lati_gofs_slice_end = 287
lat_slice_ex_gofs = example_gofs.lat.isel(lat=slice(lati_gofs_slice_start,lati_gofs_slice_end)).values
print(lat_slice_ex_gofs[0], lat_slice_ex_gofs[-1])

In [ ]:
loni_gofs_slice_start, loni_gofs_slice_end, lati_gofs_slice_start, lati_gofs_slice_end = 224, 510, 212, 287
lon_bnds=(loni_gofs_slice_start-50, loni_gofs_slice_end+51)
lat_bnds=(lati_gofs_slice_start-50, lati_gofs_slice_end+51)
example_gofs['aice'].isel(time=0).isel(lon=slice(*lon_bnds), lat=slice(*lat_bnds)).plot()


### Concatenate forecasts files to one file per initialisation date, only selecting relevant area and buffer zone
### Rename variables to reference names

In [ ]:
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)
start_dt = datetime(2024,4,1)

def _preprocess(x, lon_bnds, lat_bnds):
    ds = xr.Dataset()
    ds['SIC'] = x['aice'].isel(lon=slice(*lon_bnds), lat=slice(*lat_bnds))
    ds['sea_ice_x_velocity'] = x['uvel'].isel(lon=slice(*lon_bnds), lat=slice(*lat_bnds))
    ds['sea_ice_y_velocity'] = x['vvel'].isel(lon=slice(*lon_bnds), lat=slice(*lat_bnds))
    ds['sea_ice_x_velocity'].attrs['standard_name'] = 'sea_ice_x_velocity'
    ds['sea_ice_y_velocity'].attrs['standard_name'] = 'sea_ice_y_velocity'
    ds['Hice'] = x['hi'].isel(lon=slice(*lon_bnds), lat=slice(*lat_bnds))
    return ds
partial_func = partial(_preprocess, \
                       lon_bnds=(loni_gofs_slice_start-50, loni_gofs_slice_end+51), \
                       lat_bnds=(lati_gofs_slice_start-50, lati_gofs_slice_end+51))

for date in [start_dt + timedelta(days=i) for i in range(0,61)]:
    date_str = date.strftime('%Y%m%d')
    file_list_gofs_date = list(glob(f"{path_gofs}/*{date_str}*.nc"))
    gofs_ds_date = xr.open_mfdataset(file_list_gofs_date, combine='by_coords', preprocess=partial_func)
    gofs_ds_date.to_netcdf(os.path.join(output_folder, f"{model_name}_{date_str}.nc"))   

In [ ]:
xr.open_dataset('NRL-GOFS_conc_forecasts/NRL-GOFS_20240421.nc')

### Simulate sea ice trajectories for each complete forecast

In [ ]:
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)
output_subfolder = os.path.join(output_folder, 'trajs_alt')
os.makedirs(output_subfolder, exist_ok=True) 
# forecast starts at 12:00
start_dt = datetime(2024,4,1,12)
reader_landmask = reader_global_landmask.Reader()

# Loop through dates
for file_date in [start_dt + timedelta(days=i) for i in range(0, 61)]:
    i = 0
    o = SeaIceDrift(loglevel=0)
    file_date_str = file_date.strftime('%Y%m%d')
    reader_gofs = reader_netCDF_CF_generic.Reader(
        os.path.join(output_folder, f"{model_name}_{file_date_str}.nc"),
        proj4='+proj=latlon'
    )

    o.add_reader([reader_landmask, reader_gofs])
    for tr_nr in [x for x in range(1, 36) if x not in [5, 33]]:
        
        lon0 = dr[tr_nr]['lon'].interp(time=file_date, method = 'nearest')
        lat0 = dr[tr_nr]['lat'].interp(time=file_date, method = 'nearest')
        time_diff = np.abs((dr[tr_nr]['lon']['time']- np.datetime64(file_date)) / np.timedelta64(1, 'h'))
        min_time_diff = np.min(time_diff).values

        if (min_time_diff<0.15) and not (np.isnan(lon0) or np.isnan(lat0)):

            o.seed_elements(lon=lon0, lat=lat0, time=file_date, origin_marker=tr_nr)
            print(f"{tr_nr} seeded")
            
            i = 1
            
            
    if i != 0:
        # run with 30 minute resolution up to end time of the forecast
        o.run(end_time=reader_gofs.end_time, time_step=1800, time_step_output=1800, outfile=os.path.join(output_subfolder, f"trajs_{model_name}_{file_date_str}.nc"))


Plot trajectory simulation to compare later with reconstruction

In [ ]:
start_dt = datetime(2024,4,16)
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)
output_subfolder = os.path.join(output_folder, 'trajs_alt')
os.makedirs(output_subfolder, exist_ok=True) 
file_date_str = start_dt.strftime('%Y%m%d')
a=xr.open_dataset(os.path.join(output_subfolder, f"trajs_{model_name}_{file_date_str}.nc"))
a['lon'].sel(trajectory=3).plot()

In [ ]:
a

### Fill in overarching dataset

In [ ]:
# Initialize variables and paths
model_name = 'NRL-GOFS'
example_model_ds = xr.open_dataset(f'example_{model_name}.nc',engine="netcdf4")
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)
output_subfolder = os.path.join(output_folder, 'trajs_alt')
os.makedirs(output_subfolder, exist_ok=True)

key_lon, key_lat, key_x, key_y = 'lon', 'lat', 'lon', 'lat'

In [ ]:
# Calculate yx_model in parallel
yx_model = {}
with ThreadPoolExecutor() as executor:
    futures = {executor.submit(calculate_tr_nr_yx_model, example_model_ds, key_lon, key_lat, key_x, key_y, dataset, tr_nr): tr_nr for tr_nr in tr_nr_values}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Calculating yx_model"):
        tr_nr, y_model, x_model = future.result()
        yx_model[tr_nr] = (y_model, x_model)
        
with open(f'yx_model_{model_name}.pkl', 'wb') as f:
    pickle.dump(yx_model,f)

In [ ]:
with open(f'yx_model_{model_name}.pkl', 'rb') as f:
    yx_model = pickle.load(f)

### Fill in overarching dataset with forecast variables

In [ ]:
# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

def process_forecasts(file_date, output_folder, model_name, key_lon, key_lat, dataset, yx_model):
    file_date_str = file_date.strftime('%Y%m%d')
    logging.info(f"Processing forecasts for date: {file_date_str}")

    # Try opening forecasts
    file_name_fc = os.path.join(output_folder, f"{model_name}_{file_date_str}.nc")
    try:
        ds_fc = xr.open_dataset(file_name_fc)
        
        ds_fc['Uice'] = ds_fc['sea_ice_x_velocity']
        ds_fc['Vice'] = ds_fc['sea_ice_y_velocity']

        time_ds = dataset.time_ds
        updated_dataset = dataset.copy()

        for tr_nr, (y_model, x_model) in yx_model.items():
            mask_nan = ~(np.isnan(x_model) + np.isnan(y_model))
            for i in range(0, 240, 24):
                
                lt_int = f']{i}_{i+24}]'
                left_lt = ((ds_fc.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))> i+0.1
                right_lt = ((ds_fc.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))<= i+24+0.1
                sel_lt = np.where(left_lt*right_lt)[0]
                ds_fc_lt_int = ds_fc.isel(time=sel_lt)
                av_times_fc = ds_fc_lt_int['time'].values
                mask_av = pd.to_datetime(time_ds).isin(pd.to_datetime(av_times_fc))

                if np.sum(mask_av * mask_nan) > 0:
                    for var in ds_fc.data_vars:
                        if var not in ['sea_ice_x_velocity', 'sea_ice_y_velocity']:
                            var_model_lt_int_path = ds_fc[var].sel({'time': time_ds[mask_av * mask_nan], key_x: x_model[mask_av * mask_nan], key_y: y_model[mask_av * mask_nan]})
                            updated_dataset[var].loc[dict(model=model_name, tr_nr=tr_nr, lt_int=lt_int, time_ds=time_ds[mask_av * mask_nan])] = var_model_lt_int_path.values

        return updated_dataset
    
    except FileNotFoundError:
        logging.warning(f"File not found: {file_name_fc}")
        return dataset
    
                                      
# Process forecasts in parallel
with ThreadPoolExecutor() as executor:
    futures = [
        executor.submit(partial(process_forecasts, file_date, output_folder, model_name, key_lon, key_lat, dataset, yx_model))
        for file_date in tqdm(file_dates, desc="Processing forecasts")
    ]
    for future in tqdm(futures, desc="Updating dataset with forecast results"):
        result = future.result()
        if result is not None:
            dataset.update(result)

### Fill in overarching dataset with simulated drift trajectories

In [ ]:
def process_trajectories(file_date, output_subfolder, model_name, dataset):
    file_date_str = file_date.strftime('%Y%m%d')
    logging.info(f"Processing trajectories for date: {file_date_str}")

    file_name_traj = os.path.join(output_subfolder, f"trajs_{model_name}_{file_date_str}.nc")
    try:
        ds_traj = xr.open_dataset(file_name_traj)

        updated_dataset = dataset.copy()
        trajectory_sim = 0

        for tr_nr in np.unique(ds_traj['origin_marker'].values):
            if not np.isnan(tr_nr):
                trajectory_sim += 1
                for i in range(0, 240, 24):
                    lt_int = f']{i}_{i + 24}]'
                    left_lt = ((ds_traj.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))> i+0.1
                    right_lt = ((ds_traj.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))<= i+24+0.1
                    sel_lt = np.where(left_lt*right_lt)[0]                
                    time_traj = ds_traj.sel(trajectory=trajectory_sim)['time'].isel(time=sel_lt)
                    time_traj = time_traj.sel(time = (time_traj.time <= np.datetime64('2024-06-01T00:00')))
                    if len(time_traj.values)>0:
                        updated_dataset['lon_sim'].loc[dict(model=model_name, tr_nr=tr_nr, lt_int=lt_int, time_ds=time_traj)] = ds_traj['lon'].sel(trajectory=trajectory_sim, time=time_traj)
                        updated_dataset['lat_sim'].loc[dict(model=model_name, tr_nr=tr_nr, lt_int=lt_int, time_ds=time_traj)] = ds_traj['lat'].sel(trajectory=trajectory_sim, time=time_traj)

        return updated_dataset
    
    except FileNotFoundError:
        logging.warning(f"File not found: {file_name_traj}")
        return dataset

# Process trajectories in parallel
with ThreadPoolExecutor() as executor:
    futures = [
        executor.submit(partial(process_trajectories, file_date, output_subfolder, model_name, dataset))
        for file_date in tqdm(file_dates, desc="Processing trajectories")
    ]
    for future in tqdm(futures, desc="Updating dataset with trajectory results"):
        result = future.result()
        if result is not None:
            dataset.update(result)


### Plot to check data

In [ ]:
buoy_nr = 3
buoy_data = dataset.sel(tr_nr = buoy_nr)

colors = {
    'NERSC-NextSIM': 'tab:blue',
    'ECCC-RIOPS': 'tab:orange',
    'ECCC-RIOPScoupled': 'tab:green',
    'NRL-GOFS': 'tab:red',
    'MET-TOPAZ5': 'tab:purple',
    'MET-ROMS_m00': 'tab:cyan',
    'MET-ROMS_m01': 'tab:pink'
}

#['NERSC-NextSIM', 'ECCC-RIOPS', 'ECCC-RIOPScoupled', 'NRL-GOFS','MET-TOPAZ5', 'MET-ROMS_m00', 'MET-ROMS_m01']
#models_sel = []
models_sel = ['NRL-GOFS']
target_date = '2024-04-16'  # Example date in YYYY-MM-DD format

lon_obs_b, lat_obs_b = buoy_data['lon_obs'], buoy_data['lat_obs']
plt.figure()
start = pd.to_datetime(target_date) + pd.DateOffset(seconds=1)
end = start + pd.DateOffset(days=8)
        
lon_obs_b.sel(time_ds=slice(start,end)).plot(color='k')
for model_name in models_sel:
    color = colors.get(model_name, 'black')
    
    # Plot dummy line for legend
    plt.plot([], [], color=color, label=model_name)
    
    for k in range(0,8):
        lt_int_sel = dataset['lt_int'].values[k]
        start = pd.to_datetime(target_date) + pd.DateOffset(seconds=1) + pd.DateOffset(days=k) 
        end = start + pd.DateOffset(days=1)
        lon_sim_lt_k = buoy_data['lon_sim'].sel(model=model_name, time_ds=slice(start,end), lt_int=lt_int_sel)
        lat_sim_lt_k = buoy_data['lat_sim'].sel(model=model_name, time_ds=slice(start,end), lt_int=lt_int_sel)
        lon_sim_lt_k.plot(color=color)

plt.legend(loc='upper left')
plt.grid(True)

In [ ]:
model_name='NRL-GOFS'
dataset['lat_obs'].sel(tr_nr=21).plot()
dataset['lat_sim'].sel(tr_nr=21, model=model_name, lt_int=']0_24]').dropna(dim='time_ds', how='any').plot()
dataset['lat_sim'].sel(tr_nr=21, model=model_name, lt_int=']24_48]').dropna(dim='time_ds', how='any').plot()

In [ ]:
dataset['SIC'].sel(tr_nr=2, model='NRL-GOFS', lt_int=']0_24]').dropna(dim='time_ds', how='any').plot()

## ECCC-RIOPS

In [ ]:
model_name = 'ECCC-RIOPS'

In [ ]:
# ECCC - RIOPS
# Coordinates: time, yc, xc
# Variables: itzocrtx (sea ice x velocity [m/s]), itmecrty (sea ice y velocity [m/s]),
# iicevol (Sea ice volume per unit grid cell area [m]), isnowvol (Volume of snow on sea ice per unit grid cell area [cm])

# time: one time stamp per file
# initialization in file name
example_riops = xr.open_dataset('/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/ECCC-RIOPS/2024050600_084_2D_ps5km60N.nc',engine="netcdf4")


In [ ]:
example_riops.to_netcdf(f'example_{model_name}.nc')

In [ ]:
example_riops = xr.open_dataset('example_ECCC-RIOPS.nc')
example_riops

### Calculate local orientation of grid to rotate sea ice drift vectors

In [ ]:
# Variables 
xx = example_riops.xc
yy = example_riops.yc
lat = example_riops.latitude
lon = example_riops.longitude
alphaa=lat #Target matrix

for j in range(0,yy.size-1):
    for i in range(0,xx.size-1):
        # Prevent out of bounds
        if j==yy.size-1:
            j1=j-1; j2=j
        else:
            j1=j; j2=j+1
        if i==xx.size-1:
            i1=i-1; i2=i
        else:
            i1=i; i2=i+1

        dlatykm=distance([lat[j1,i1],lon[j1,i1]],[lat[j2,i1],lon[j1,i1]])
        dlonykm=distance([lat[j1,i1],lon[j1,i1]],[lat[j1,i1],lon[j2,i1]])
        
        alphaa[j,i]=np.arctan2(dlatykm,dlonykm)*180/np.pi - 90
alphaa_ds = xr.DataArray(alphaa, dims=['yc', 'xc'])
alphaa_ds.to_netcdf('alpha_'+model_name+'.nc')

### Create list of files of interest

In [ ]:
path_riops = "/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/ECCC-RIOPS/"
file_list_riops = list(glob(path_riops + "*60N.nc"))

file_list_riops.sort()
file_list_riops

In [ ]:
file_list_riops[-1]

### Determine relevant grid area + buffer zone

In [ ]:
used_area_riops = np.zeros(example_riops['latitude'].values.shape)

for tr_nr in [x for x in range(1, 36) if x not in [5, 33]]:
    # loop through each position, indicate closest point in used_area_riops
    for time in dr[tr_nr]['time']:
        lat_time = dr[tr_nr]['lat'].sel(time=time).values
        lon_time = dr[tr_nr]['lon'].sel(time=time).values
        
        c_riops = np.maximum(np.abs(example_riops['latitude'].values-lat_time), np.abs(example_riops['longitude'].values-lon_time))
        used_area_riops[c_riops == np.min(c_riops)] = 1


In [ ]:
yi_riops_min, xi_riops_min = np.min(np.where(used_area_riops == 1), axis=1)
yi_riops_max, xi_riops_max = np.max(np.where(used_area_riops == 1), axis=1)
print(yi_riops_min, xi_riops_min)
print(yi_riops_max, xi_riops_max)

In [ ]:
yi_riops_min, xi_riops_min, yi_riops_max, xi_riops_max = 1095, 1005, 1169, 1087


In [ ]:
# add buffer for sea ice drift near edge
yi_riops_min -= 50
xi_riops_min -= 50
yi_riops_max += 50
xi_riops_max += 50
plt.imshow(example_riops['iicevol'].isel(time=0).isel(xc=slice(xi_riops_min,xi_riops_max+1), yc=slice(yi_riops_min,yi_riops_max+1)))


### Concatenate forecasts files to one file per initialisation date, only selecting relevant area and buffer zone
### Rename variables to reference names

In [ ]:
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)

start_dt = datetime(2024,4,7)
def _preprocess(x, x_bnds, y_bnds):
    ds = xr.Dataset()
    ds['SIC'] = x['iiceconc'].isel(xc=slice(*x_bnds), yc=slice(*y_bnds))
    ds['sea_ice_x_velocity'] = x['itzocrtx'].isel(xc=slice(*x_bnds), yc=slice(*y_bnds))
    ds['sea_ice_y_velocity'] = x['itmecrty'].isel(xc=slice(*x_bnds), yc=slice(*y_bnds))
    ds['Hice'] = x['iicevol'].isel(xc=slice(*x_bnds), yc=slice(*y_bnds))
    ds['Sd'] = x['isnowvol'].isel(xc=slice(*x_bnds), yc=slice(*y_bnds))
    return ds
partial_func = partial(_preprocess, \
                       x_bnds=(xi_riops_min, xi_riops_max+1), \
                       y_bnds=(yi_riops_min, yi_riops_max+1))

for date in [start_dt + timedelta(days=i) for i in range(18,30)]:
    date_str = date.strftime('%Y%m%d')
    print(date_str)
    file_list_riops_date = list(glob(f"{path_riops}/{date_str}*.nc"))
    file_list_riops_date.sort()
    riops_ds_date=xr.open_mfdataset(file_list_riops_date, combine='by_coords', preprocess=partial_func)
    riops_ds_date.to_netcdf(os.path.join(output_folder, f"{model_name}_{date_str}.nc")) 
    
    

In [ ]:
xr.open_dataset('ECCC-RIOPS_conc_forecasts/ECCC-RIOPS_20240408.nc')

### Simulate sea ice trajectories for each complete forecast

In [ ]:
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)
output_subfolder = os.path.join(output_folder, 'trajs_alt')
os.makedirs(output_subfolder, exist_ok=True) 
# forecast starts at 00:00
start_dt = datetime(2024,4,7)
reader_landmask = reader_global_landmask.Reader()

# Loop through dates
for file_date in [start_dt + timedelta(hours=i) for i in range(0, 30*24,24)]:
    i = 0
    o = SeaIceDrift(loglevel=0)
    file_date_str = file_date.strftime('%Y%m%d')
    ds_riops = xr.open_dataset(os.path.join(output_folder, f"{model_name}_{file_date_str}.nc"))
    #ds_riops['sea_ice_x_velocity'] = ds_riops['uui']*np.cos(alpha/180*np.pi) + ds_riops['vvi']*np.sin(alpha/180*np.pi)
    #ds_riops['sea_ice_y_velocity'] = -ds_riops['uui']*np.sin(alpha/180*np.pi) + ds_riops['vvi']*np.cos(alpha/180*np.pi)
    #ds_riops['sea_ice_x_velocity'].attrs['standard_name'] = 'sea_ice_x_velocity'
    #ds_riops['sea_ice_y_velocity'].attrs['standard_name'] = 'sea_ice_y_velocity'
    
    reader_riops = reader_netCDF_CF_generic.Reader(ds_riops, \
        proj4 = '+proj=stere +lat_0=90.0 +lon_0=-100.0 +k=0.9330124 +x_0=4245000.0 +y_0=5295000.0 +a=6371229.0 +b=6371229.0')


    o.add_reader([reader_landmask, reader_riops])
    for tr_nr in [x for x in range(1, 36) if x not in [5, 33]]:
        
        lon0 = dr[tr_nr]['lon'].interp(time=file_date, method = 'nearest')
        lat0 = dr[tr_nr]['lat'].interp(time=file_date, method = 'nearest')
        time_diff = np.abs((dr[tr_nr]['lon']['time']- np.datetime64(file_date)) / np.timedelta64(1, 'h'))
        min_time_diff = np.min(time_diff).values

        if (min_time_diff<0.15) and not (np.isnan(lon0) or np.isnan(lat0)):

            o.seed_elements(lon=lon0, lat=lat0, time=file_date, origin_marker=tr_nr)
            print(f"{tr_nr} seeded")
            
            i = 1
            
    if i != 0:
        # run with 30 minute resolution up to end time of the forecast
        o.run(end_time=reader_riops.end_time, time_step=1800, time_step_output=1800, outfile=os.path.join(output_subfolder, f"trajs_{model_name}_{file_date_str}.nc"))


Plot trajectory simulation to compare later with reconstruction

In [ ]:
start_dt = datetime(2024,4,16)
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)
output_subfolder = os.path.join(output_folder, 'trajs_alt')
os.makedirs(output_subfolder, exist_ok=True) 
file_date_str = start_dt.strftime('%Y%m%d')
a=xr.open_dataset(os.path.join(output_subfolder, f"trajs_{model_name}_{file_date_str}.nc"))
a['lon'].sel(trajectory=3).plot()

In [ ]:
a

### Fill in overarching dataset

In [ ]:
# Initialize variables and paths
model_name = 'ECCC-RIOPS'
example_model_ds = xr.open_dataset(f'example_{model_name}.nc',engine="netcdf4")
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)
output_subfolder = os.path.join(output_folder, 'trajs_alt')
os.makedirs(output_subfolder, exist_ok=True)

key_lon, key_lat, key_x, key_y = 'longitude', 'latitude', 'xc', 'yc'
alpha_ds = xr.open_dataset('alpha_' + model_name + '.nc')

In [ ]:
# Calculate yx_model in parallel
yx_model = {}
with ThreadPoolExecutor() as executor:
    futures = {executor.submit(calculate_tr_nr_yx_model, example_model_ds, key_lon, key_lat, key_x, key_y, dataset, tr_nr): tr_nr for tr_nr in tr_nr_values}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Calculating yx_model"):
        tr_nr, y_model, x_model = future.result()
        yx_model[tr_nr] = (y_model, x_model)
        
with open(f'yx_model_{model_name}.pkl', 'wb') as f:
    pickle.dump(yx_model,f)

In [ ]:
with open(f'yx_model_{model_name}.pkl', 'rb') as f:
    yx_model = pickle.load(f)

### Fill in overarching dataset with forecast variables

In [ ]:
# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

def process_forecasts(file_date, output_folder, model_name, key_lon, key_lat, dataset, yx_model):
    path_riops = "/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/ECCC-RIOPS/"
    file_date_str = file_date.strftime('%Y%m%d')
    logging.info(f"Processing forecasts for date: {file_date_str}")

    # Try opening forecasts
    try:
        ds_fc = xr.open_dataset(os.path.join(output_folder, f"{model_name}_{file_date_str}.nc"))

        
        alpha_rad = alpha_ds['__xarray_dataarray_variable__'] * np.pi / 180
        ds_fc['Uice'] = ds_fc['sea_ice_x_velocity'] * np.cos(alpha_rad) - ds_fc['sea_ice_y_velocity'] * np.sin(alpha_rad)
        ds_fc['Vice'] = ds_fc['sea_ice_x_velocity'] * np.sin(alpha_rad) + ds_fc['sea_ice_y_velocity'] * np.cos(alpha_rad)
    

        updated_dataset = dataset.copy()
        time_ds = dataset.time_ds

        for tr_nr, (y_model, x_model) in yx_model.items():
            mask_nan = ~(np.isnan(x_model) + np.isnan(y_model))
            for i in range(0, 240, 24):
                lt_int = f']{i}_{i+24}]'
                left_lt = ((ds_fc.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))> i+0.1
                right_lt = ((ds_fc.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))<= i+24+0.1
                sel_lt = np.where(left_lt*right_lt)[0]
                ds_fc_lt_int = ds_fc.isel(time=sel_lt)
                av_times_fc = ds_fc_lt_int['time'].values
                mask_av = pd.to_datetime(time_ds).isin(pd.to_datetime(av_times_fc))

                if np.sum(mask_av * mask_nan) > 0:
                    for var in ds_fc.data_vars:
                        if var not in ['sea_ice_x_velocity', 'sea_ice_y_velocity']:
                            var_model_lt_int_path = ds_fc[var].sel({'time': time_ds[mask_av * mask_nan], key_x: x_model[mask_av * mask_nan].astype(int), key_y: y_model[mask_av * mask_nan].astype(int)})
                            updated_dataset[var].loc[dict(model=model_name, tr_nr=tr_nr, lt_int=lt_int, time_ds=time_ds[mask_av * mask_nan])] = var_model_lt_int_path.values
        return updated_dataset

    except FileNotFoundError:
        logging.warning(f"File not found: "+ str(os.path.join(output_folder, f"{model_name}_{file_date_str}.nc")))
        return dataset
        

    

# Process forecasts in parallel
with ThreadPoolExecutor() as executor:
    futures = [
        executor.submit(partial(process_forecasts, file_date, output_folder, model_name, key_lon, key_lat, dataset, yx_model))
        for file_date in tqdm(file_dates, desc="Processing forecasts")
    ]
    for future in tqdm(futures, desc="Updating dataset with forecast results"):
        result = future.result()
        if result is not None:
            dataset.update(result)


### Fill in overarching dataset with simulated drift trajectories

In [ ]:
def process_trajectories(file_date, output_subfolder, model_name, dataset):
    file_date_str = file_date.strftime('%Y%m%d')
    logging.info(f"Processing trajectories for date: {file_date_str}")

    file_name_traj = os.path.join(output_subfolder, f"trajs_{model_name}_{file_date_str}.nc")
    try:
        ds_traj = xr.open_dataset(file_name_traj)
    
        updated_dataset = dataset.copy()
        trajectory_sim = 0

        for tr_nr in np.unique(ds_traj['origin_marker'].values):
            if not np.isnan(tr_nr):
                trajectory_sim += 1
                for i in range(0, 240, 24):
                    lt_int = f']{i}_{i + 24}]'
                    left_lt = ((ds_traj.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))> i+0.1
                    right_lt = ((ds_traj.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))<= i+24+0.1
                    sel_lt = np.where(left_lt*right_lt)[0]                
                    time_traj = ds_traj.sel(trajectory=trajectory_sim)['time'].isel(time=sel_lt)
                    time_traj = time_traj.sel(time = (time_traj.time <= np.datetime64('2024-06-01T00:00')))
                    if len(time_traj.values)>0:
                        updated_dataset['lon_sim'].loc[dict(model=model_name, tr_nr=tr_nr, lt_int=lt_int, time_ds=time_traj)] = ds_traj['lon'].sel(trajectory=trajectory_sim, time=time_traj)
                        updated_dataset['lat_sim'].loc[dict(model=model_name, tr_nr=tr_nr, lt_int=lt_int, time_ds=time_traj)] = ds_traj['lat'].sel(trajectory=trajectory_sim, time=time_traj)

        return updated_dataset
        
    except FileNotFoundError:
        logging.warning(f"File not found: {file_name_traj}")
        return dataset
    
    

    
# Process trajectories in parallel
with ThreadPoolExecutor() as executor:
    futures = [
        executor.submit(partial(process_trajectories, file_date, output_subfolder, model_name, dataset))
        for file_date in tqdm(file_dates, desc="Processing trajectories")
    ]
    for future in tqdm(futures, desc="Updating dataset with trajectory results"):
        result = future.result()
        if result is not None:
            dataset.update(result)


In [ ]:
dataset.to_netcdf('traj_ds_RIOPS.nc')

### Plot to check data

In [ ]:
buoy_nr = 3
buoy_data = dataset.sel(tr_nr = buoy_nr)

colors = {
    'NERSC-NextSIM': 'tab:blue',
    'ECCC-RIOPS': 'tab:orange',
    'ECCC-RIOPScoupled': 'tab:green',
    'NRL-GOFS': 'tab:red',
    'MET-TOPAZ5': 'tab:purple',
    'MET-ROMS_m00': 'tab:cyan',
    'MET-ROMS_m01': 'tab:pink'
}

#['NERSC-NextSIM', 'ECCC-RIOPS', 'ECCC-RIOPScoupled', 'NRL-GOFS','MET-TOPAZ5', 'MET-ROMS_m00', 'MET-ROMS_m01']
#models_sel = []
models_sel = ['ECCC-RIOPS']
target_date = '2024-04-16'  # Example date in YYYY-MM-DD format

lon_obs_b, lat_obs_b = buoy_data['lon_obs'], buoy_data['lat_obs']
plt.figure()
start = pd.to_datetime(target_date) + pd.DateOffset(seconds=1)
end = start + pd.DateOffset(days=4)
        
lon_obs_b.sel(time_ds=slice(start,end)).plot(color='k')
for model_name in models_sel:
    color = colors.get(model_name, 'black')
    
    # Plot dummy line for legend
    plt.plot([], [], color=color, label=model_name)
    
    for k in range(0,4):
        lt_int_sel = dataset['lt_int'].values[k]
        start = pd.to_datetime(target_date) + pd.DateOffset(seconds=1) + pd.DateOffset(days=k) 
        end = start + pd.DateOffset(days=1)
        lon_sim_lt_k = buoy_data['lon_sim'].sel(model=model_name, time_ds=slice(start,end), lt_int=lt_int_sel)
        lat_sim_lt_k = buoy_data['lat_sim'].sel(model=model_name, time_ds=slice(start,end), lt_int=lt_int_sel)
        lon_sim_lt_k.plot(color=color)

plt.legend(loc='upper left')
plt.grid(True)

In [ ]:
model_name='ECCC-RIOPS'
dataset['SIC'].sel(tr_nr=4, model=model_name, lt_int=']0_24]').dropna(dim='time_ds', how='any').plot()

In [ ]:
tr_nr=8
dataset['lat_obs'].sel(tr_nr=tr_nr).plot()
dataset['lat_sim'].sel(tr_nr=tr_nr, model=model_name, lt_int=']0_24]').dropna(dim='time_ds', how='any').plot()
dataset['lat_sim'].sel(tr_nr=tr_nr, model=model_name, lt_int=']24_48]').dropna(dim='time_ds', how='any').plot()

## 

## ECCC-RIOPScoupled

In [ ]:
model_name = 'ECCC-RIOPScoupled'

In [ ]:
# ECCC - RIOPS
# Coordinates: time, yc, xc
# Variables: itzocrtx (sea ice x velocity [m/s]), itmecrty (sea ice y velocity [m/s]),
# iicevol (Sea ice volume per unit grid cell area [m]), isnowvol (Volume of snow on sea ice per unit grid cell area [cm])

# time: one time stamp per file
# initialization in file name
example_riops_coupled = xr.open_dataset('/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/ECCC-RIOPScoupled/2024041600_027_2D_ps5km60N.nc',engine="netcdf4")


In [ ]:
example_riops_coupled

### Create list of files of interest

In [ ]:
path_riops_coupled = "/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/ECCC-RIOPScoupled/"
file_list_riops_coupled = list(glob(path_riops_coupled + "*60N.nc"))

file_list_riops_coupled.sort()
file_list_riops_coupled

In [ ]:
file_list_riops_coupled[-1]

### Determine relevant grid area + buffer zone

idem grid as RIOPS

In [ ]:
yi_riops_min, xi_riops_min, yi_riops_max, xi_riops_max = 1095, 1005, 1169, 1087

In [ ]:
# add buffer for sea ice drift near edge
yi_riops_min -= 50
xi_riops_min -= 50
yi_riops_max += 50
xi_riops_max += 50
plt.imshow(example_riops_coupled['iicevol'].isel(time=0).isel(xc=slice(xi_riops_min,xi_riops_max+1), yc=slice(yi_riops_min,yi_riops_max+1)))


### Concatenate forecasts files to one file per initialisation date, only selecting relevant area and buffer zone
### Rename variables to reference names

In [ ]:
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)

start_dt = datetime(2024,4,1)
def _preprocess(x, x_bnds, y_bnds):
    ds = xr.Dataset()
    ds['SIC'] = x['iiceconc'].isel(xc=slice(*x_bnds), yc=slice(*y_bnds))
    ds['sea_ice_x_velocity'] = x['itzocrtx'].isel(xc=slice(*x_bnds), yc=slice(*y_bnds))
    ds['sea_ice_y_velocity'] = x['itmecrty'].isel(xc=slice(*x_bnds), yc=slice(*y_bnds))
    ds['Hice'] = x['iicevol'].isel(xc=slice(*x_bnds), yc=slice(*y_bnds))
    ds['Sd'] = x['isnowvol'].isel(xc=slice(*x_bnds), yc=slice(*y_bnds))
    return ds
partial_func = partial(_preprocess, \
                       x_bnds=(xi_riops_min, xi_riops_max+1), \
                       y_bnds=(yi_riops_min, yi_riops_max+1))

for date in [start_dt + timedelta(days=i) for i in range(0,36)]:
    date_str = date.strftime('%Y%m%d')
    print(date_str)
    file_list_riops_coupled_date = list(glob(f"{path_riops_coupled}/{date_str}*.nc"))
    file_list_riops_coupled_date.sort()
    riops_coupled_ds_date=xr.open_mfdataset(file_list_riops_coupled_date, combine='by_coords', preprocess=partial_func)
    riops_coupled_ds_date.to_netcdf(os.path.join(output_folder, f"{model_name}_{date_str}.nc")) 
    

In [ ]:
xr.open_dataset('ECCC-RIOPScoupled_conc_forecasts/ECCC-RIOPScoupled_20240402.nc')

### Simulate sea ice trajectories for each complete forecast

In [ ]:
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)
output_subfolder = os.path.join(output_folder, 'trajs_alt')
os.makedirs(output_subfolder, exist_ok=True) 
# forecast starts at 01:00
start_dt = datetime(2024,4,1,1)
reader_landmask = reader_global_landmask.Reader()

# Loop through dates
for file_date in [start_dt + timedelta(hours=i) for i in range(0, 36*24,24)]:
    i = 0
    o = SeaIceDrift(loglevel=0)
    file_date_str = file_date.strftime('%Y%m%d')
    ds_riops_coupled = xr.open_dataset(os.path.join(output_folder, f"{model_name}_{file_date_str}.nc"))
    reader_riops_coupled = reader_netCDF_CF_generic.Reader(ds_riops_coupled, \
        proj4 = '+proj=stere +lat_0=90.0 +lon_0=-100.0 +k=0.9330124 +x_0=4245000.0 +y_0=5295000.0 +a=6371229.0 +b=6371229.0')
    o.add_reader([reader_landmask, reader_riops_coupled])
    for tr_nr in [x for x in range(1, 36) if x not in [5, 33]]:
        
        lon0 = dr[tr_nr]['lon'].interp(time=file_date, method = 'nearest')
        lat0 = dr[tr_nr]['lat'].interp(time=file_date, method = 'nearest')
        time_diff = np.abs((dr[tr_nr]['lon']['time']- np.datetime64(file_date)) / np.timedelta64(1, 'h'))
        min_time_diff = np.min(time_diff).values

        if (min_time_diff<0.15) and not (np.isnan(lon0) or np.isnan(lat0)):

            o.seed_elements(lon=lon0, lat=lat0, time=file_date, origin_marker=tr_nr)
            print(f"{tr_nr} seeded")
            
            i = 1
            
    if i != 0:
        # run with 30 minute resolution up to end time of the forecast
        o.run(end_time=reader_riops_coupled.end_time, time_step=1800, time_step_output=1800, outfile=os.path.join(output_subfolder, f"trajs_{model_name}_{file_date_str}.nc"))


Plot trajectory simulation to compare later with reconstruction

In [ ]:
start_dt = datetime(2024,4,16)
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)
output_subfolder = os.path.join(output_folder, 'trajs_alt')
os.makedirs(output_subfolder, exist_ok=True) 
file_date_str = start_dt.strftime('%Y%m%d')
a=xr.open_dataset(os.path.join(output_subfolder, f"trajs_{model_name}_{file_date_str}.nc"))
a['lon'].sel(trajectory=3).plot()

In [ ]:
a

### Fill in overarching dataset

In [ ]:
# Initialize variables and paths
model_name = 'ECCC-RIOPScoupled'
example_model_ds = xr.open_dataset(f'example_ECCC-RIOPS.nc',engine="netcdf4")
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)
output_subfolder = os.path.join(output_folder, 'trajs_alt')
os.makedirs(output_subfolder, exist_ok=True)

key_lon, key_lat, key_x, key_y = 'longitude', 'latitude', 'xc', 'yc'
alpha_ds = xr.open_dataset('alpha_ECCC-RIOPS.nc')
with open(f'yx_model_ECCC-RIOPS.pkl', 'rb') as f:
    yx_model = pickle.load(f)

### Fill in overarching dataset with forecast variables

In [ ]:
# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

def process_forecasts(file_date, output_folder, model_name, key_lon, key_lat, dataset, yx_model):
    path_riops = "/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/ECCC-RIOPScoupled/"
    file_date_str = file_date.strftime('%Y%m%d')
    logging.info(f"Processing forecasts for date: {file_date_str}")

    # Try opening forecasts
    try:
        ds_fc = xr.open_dataset(os.path.join(output_folder, f"{model_name}_{file_date_str}.nc"))

        
        alpha_rad = alpha_ds['__xarray_dataarray_variable__'] * np.pi / 180
        ds_fc['Uice'] = ds_fc['sea_ice_x_velocity'] * np.cos(alpha_rad) - ds_fc['sea_ice_y_velocity'] * np.sin(alpha_rad)
        ds_fc['Vice'] = ds_fc['sea_ice_x_velocity'] * np.sin(alpha_rad) + ds_fc['sea_ice_y_velocity'] * np.cos(alpha_rad)
    

        updated_dataset = dataset.copy()
        time_ds = dataset.time_ds

        for tr_nr, (y_model, x_model) in yx_model.items():
            mask_nan = ~(np.isnan(x_model) + np.isnan(y_model))
            for i in range(0, 240, 24):
                lt_int = f']{i}_{i+24}]'
                left_lt = ((ds_fc.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))> i+0.1
                right_lt = ((ds_fc.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))<= i+24+0.1
                sel_lt = np.where(left_lt*right_lt)[0]
                ds_fc_lt_int = ds_fc.isel(time=sel_lt)
                av_times_fc = ds_fc_lt_int['time'].values
                mask_av = pd.to_datetime(time_ds).isin(pd.to_datetime(av_times_fc))

                if np.sum(mask_av * mask_nan) > 0:
                    for var in ds_fc.data_vars:
                        if var not in ['sea_ice_x_velocity', 'sea_ice_y_velocity']:
                            var_model_lt_int_path = ds_fc[var].sel({'time': time_ds[mask_av * mask_nan], key_x: x_model[mask_av * mask_nan].astype(int), key_y: y_model[mask_av * mask_nan].astype(int)})
                            updated_dataset[var].loc[dict(model=model_name, tr_nr=tr_nr, lt_int=lt_int, time_ds=time_ds[mask_av * mask_nan])] = var_model_lt_int_path.values
        return updated_dataset

    except FileNotFoundError:
        logging.warning(f"File not found: "+ str(os.path.join(output_folder, f"{model_name}_{file_date_str}.nc")))
        return dataset
        

    

# Process forecasts in parallel
with ThreadPoolExecutor() as executor:
    futures = [
        executor.submit(partial(process_forecasts, file_date, output_folder, model_name, key_lon, key_lat, dataset, yx_model))
        for file_date in tqdm(file_dates, desc="Processing forecasts")
    ]
    for future in tqdm(futures, desc="Updating dataset with forecast results"):
        result = future.result()
        if result is not None:
            dataset.update(result)


### Fill in overarching dataset with simulated drift trajectories

In [ ]:
def process_trajectories(file_date, output_subfolder, model_name, dataset):
    file_date_str = file_date.strftime('%Y%m%d')
    logging.info(f"Processing trajectories for date: {file_date_str}")

    file_name_traj = os.path.join(output_subfolder, f"trajs_{model_name}_{file_date_str}.nc")
    try:
        ds_traj = xr.open_dataset(file_name_traj)
    
        updated_dataset = dataset.copy()
        trajectory_sim = 0

        for tr_nr in np.unique(ds_traj['origin_marker'].values):
            if not np.isnan(tr_nr):
                trajectory_sim += 1
                for i in range(0, 240, 24):
                    lt_int = f']{i}_{i + 24}]'
                    left_lt = ((ds_traj.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))> i+0.1
                    right_lt = ((ds_traj.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))<= i+24+0.1
                    sel_lt = np.where(left_lt*right_lt)[0]                
                    time_traj = ds_traj.sel(trajectory=trajectory_sim)['time'].isel(time=sel_lt)
                    time_traj = time_traj.sel(time = (time_traj.time <= np.datetime64('2024-06-01T00:00')))
                    if len(time_traj.values)>0:
                        updated_dataset['lon_sim'].loc[dict(model=model_name, tr_nr=tr_nr, lt_int=lt_int, time_ds=time_traj)] = ds_traj['lon'].sel(trajectory=trajectory_sim, time=time_traj)
                        updated_dataset['lat_sim'].loc[dict(model=model_name, tr_nr=tr_nr, lt_int=lt_int, time_ds=time_traj)] = ds_traj['lat'].sel(trajectory=trajectory_sim, time=time_traj)

        return updated_dataset
        
    except FileNotFoundError:
        logging.warning(f"File not found: {file_name_traj}")
        return dataset
    
    

    
# Process trajectories in parallel
with ThreadPoolExecutor() as executor:
    futures = [
        executor.submit(partial(process_trajectories, file_date, output_subfolder, model_name, dataset))
        for file_date in tqdm(file_dates, desc="Processing trajectories")
    ]
    for future in tqdm(futures, desc="Updating dataset with trajectory results"):
        result = future.result()
        if result is not None:
            dataset.update(result)


In [ ]:
dataset.to_netcdf('traj_ds_RIOPScoupled.nc')

### Plot to check data

In [ ]:
buoy_nr = 3
buoy_data = dataset.sel(tr_nr = buoy_nr)

colors = {
    'NERSC-NextSIM': 'tab:blue',
    'ECCC-RIOPS': 'tab:orange',
    'ECCC-RIOPScoupled': 'tab:green',
    'NRL-GOFS': 'tab:red',
    'MET-TOPAZ5': 'tab:purple',
    'MET-ROMS_m00': 'tab:cyan',
    'MET-ROMS_m01': 'tab:pink'
}

#['NERSC-NextSIM', 'ECCC-RIOPS', 'ECCC-RIOPScoupled', 'NRL-GOFS','MET-TOPAZ5', 'MET-ROMS_m00', 'MET-ROMS_m01']
#models_sel = []
models_sel = ['ECCC-RIOPScoupled']
target_date = '2024-04-16'  # Example date in YYYY-MM-DD format

lon_obs_b, lat_obs_b = buoy_data['lon_obs'], buoy_data['lat_obs']
plt.figure()
start = pd.to_datetime(target_date) + pd.DateOffset(seconds=1)
end = start + pd.DateOffset(days=2)
        
lon_obs_b.sel(time_ds=slice(start,end)).plot(color='k')
for model_name in models_sel:
    color = colors.get(model_name, 'black')
    
    # Plot dummy line for legend
    plt.plot([], [], color=color, label=model_name)
    
    for k in range(0,2):
        lt_int_sel = dataset['lt_int'].values[k]
        start = pd.to_datetime(target_date) + pd.DateOffset(seconds=1) + pd.DateOffset(days=k) 
        end = start + pd.DateOffset(days=1)
        lon_sim_lt_k = buoy_data['lon_sim'].sel(model=model_name, time_ds=slice(start,end), lt_int=lt_int_sel)
        lat_sim_lt_k = buoy_data['lat_sim'].sel(model=model_name, time_ds=slice(start,end), lt_int=lt_int_sel)
        lon_sim_lt_k.plot(color=color)

plt.legend(loc='upper left')
plt.grid(True)

In [ ]:
tr_nr=8
dataset['lat_obs'].sel(tr_nr=tr_nr).plot()
dataset['lat_sim'].sel(tr_nr=tr_nr, model=model_name, lt_int=']0_24]').dropna(dim='time_ds', how='any').plot()
dataset['lat_sim'].sel(tr_nr=tr_nr, model='ECCC-RIOPS', lt_int=']0_24]').dropna(dim='time_ds', how='any').plot()

In [ ]:
model_name='ECCC-RIOPScoupled'
dataset['SIC'].sel(tr_nr=4, model=model_name, lt_int=']0_24]').dropna(dim='time_ds', how='any').plot()
dataset['SIC'].sel(tr_nr=4, model='ECCC-RIOPS', lt_int=']0_24]').dropna(dim='time_ds', how='any').plot()
dataset['SIC'].sel(tr_nr=4, model='NRL-GOFS', lt_int=']0_24]').dropna(dim='time_ds', how='any').plot()

## MET-ROMS

The structure of the available files is the following:
1. sdepth/
barents_his_yyyymmddThhZm00_AN.nc
barents_his_yyyymmddThhZm00_FC.nc
Depth levels: all depths  (levels = 42, s)
Members: m00
Time covered: (AN) first 24 time steps + (FC) most recent forecast (the FC-file is overwritten daily)
2. zdepth/
barents_zdepth_yyyymmddT00Zm00_AN.nc
barents_zdepth_m00_FC.nc 
Depth levels: all depths (levels = 42, interpolated to z)
Members: m00
Time covered: (AN) first 24 time steps + (FC) most recent forecast (the FC-file is overwritten daily
3. surface/yyyy/mm/dd/ThhZ (based on zdepth)
barents_sfc_yyyymmddThhZm??.nc
Number of files: 24
Depth levels: only surface (levels = 1)
Members: 1 member per file
Time covered: 96hrs*
4. surface-eps-mean (based on surface) in surface/yyyy/mm/dd/ThhZ
barents2500_his_sfc_yyyymmdd_ensmean_96h.nc
Number of files: 1
Depth levels: only surface (levels = 1)
Members: all / ensemble (1)
Time covered: 96hrs*
5. surface-eps-std (based on surface) in surface/yyyy/mm/dd/ThhZ
barents2500_his_sfc_yyyymmdd_ensstd_96h.nc
Number of files: 1
Depth levels: only surface (levels = 1)
Members: all / ensemble (1)
Time covered: 96hrs*
* Total 97 timesteps including initial state


Members with m00, m06, m12, and m18 in their names are forced by AROME-Arctic for 66h of the forecast length, and for the rest of the 96h of the forecast, random members of the ECMWF ensemble are used. All the other members are forced by ECMWF atmospheric forcing through the entire forecast length.


In [ ]:
model_name = 'MET-ROMS'

In [ ]:
# MET-ROMS
# surface: file ending in m00 forced by AROME-Arctic for 66h of the forecast length
#          file ending in m01 forced by ECMWF
# based on zdepth
# new file daily at T00Z (for m00 and m01)
# Coordinates: time, Y, X
# Variables: ice_concentration, ice_u, ice_v, ice_thickness
# time = initialization time, hourly intervals
# initialization in file name
example_roms_sdepth = xr.open_dataset('/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/MET-ROMS/archive/sdepth/barents_his_20240829T00Zm00_AN.nc',engine="netcdf4")
example_roms_zdepth = xr.open_dataset('/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/MET-ROMS/archive/zdepth/barents_zdepth_20240829T00Zm00_AN.nc',engine="netcdf4")
example_roms_surface_m00 = xr.open_dataset('/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/MET-ROMS/archive/surface/2024/04/14/T00Z/barents_sfc_20240414T00Zm00.nc',engine="netcdf4")
example_roms_surface_m01 = xr.open_dataset('/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/MET-ROMS/archive/surface/2024/04/14/T00Z/barents_sfc_20240414T00Zm01.nc',engine="netcdf4")
example_roms_surface_eps_mean = xr.open_dataset('/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/MET-ROMS/archive/surface/2024/04/14/barents2500_his_sfc_20240414_ensmean_96h.nc',engine="netcdf4")
example_roms_surface_eps_std = xr.open_dataset('/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/MET-ROMS/archive/surface/2024/04/14/barents2500_his_sfc_20240414_ensstd_96h.nc',engine="netcdf4")


In [ ]:
example_roms_surface_m00

In [ ]:
example_roms_surface_m00.to_netcdf(f'example_{model_name}_m00.nc')
example_roms_surface_m01.to_netcdf(f'example_{model_name}_m01.nc')

In [ ]:
example_roms_surface_m00 = xr.open_dataset('example_MET-ROMS_m00.nc')
example_roms_surface_m01 = xr.open_dataset('example_MET-ROMS_m01.nc')

In [ ]:
example_roms_surface_m00['ice_u'].isel(time=50).plot()

### Calculate local orientation of grid to rotate sea ice drift vectors

In [ ]:
# Variables 
xx = example_roms_surface_m00.X
yy = example_roms_surface_m00.Y
lat = example_roms_surface_m00.lat
lon = example_roms_surface_m00.lon
alphaa=lat #Target matrix

for j in range(0,yy.size-1):
    for i in range(0,xx.size-1):
        # Prevent out of bounds
        if j==yy.size-1:
            j1=j-1; j2=j
        else:
            j1=j; j2=j+1
        if i==xx.size-1:
            i1=i-1; i2=i
        else:
            i1=i; i2=i+1

        dlatykm=distance([lat[j1,i1],lon[j1,i1]],[lat[j2,i1],lon[j1,i1]])
        dlonykm=distance([lat[j1,i1],lon[j1,i1]],[lat[j1,i1],lon[j2,i1]])
        
        alphaa[j,i]=np.arctan2(dlatykm,dlonykm)*180/np.pi - 90
alphaa_ds = xr.DataArray(alphaa, dims=['Y', 'X'])
alphaa_ds.to_netcdf('alpha_'+model_name+'.nc')

### Create list of files of interest

In [ ]:
path_roms_apr = '/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/MET-ROMS/archive/surface/2024/04/'
path_roms_may = '/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/MET-ROMS/archive/surface/2024/05/'

file_list_roms_apr_m00 = list(glob(path_roms_apr + "*/T00Z/*m00.nc"))
file_list_roms_may_m00 = list(glob(path_roms_may + "*/T00Z/*m00.nc"))
file_list_roms_m00 = file_list_roms_apr_m00 + file_list_roms_may_m00
file_list_roms_m00.sort()

file_list_roms_apr_m01 = list(glob(path_roms_apr + "*/T00Z/*m01.nc"))
file_list_roms_may_m01 = list(glob(path_roms_may + "*/T00Z/*m01.nc"))
file_list_roms_m01 = file_list_roms_apr_m01 + file_list_roms_may_m01
file_list_roms_m01.sort()

In [ ]:
file_list_roms_m00

### Determine relevant grid area + buffer zone

In [ ]:
used_area_roms = np.zeros(example_roms_surface_m00['lat'].values.shape)

for tr_nr in [x for x in range(1, 36) if x not in [5, 33]]:
    # loop through each position, indicate closest point in used_area_riops
    for time in dr[tr_nr]['time']:
        lat_time = dr[tr_nr]['lat'].sel(time=time).values
        lon_time = dr[tr_nr]['lon'].sel(time=time).values
        
        c_roms = np.maximum(np.abs(example_roms_surface_m00['lat'].values-lat_time), np.abs(example_roms_surface_m00['lon'].values-lon_time))
        used_area_roms[c_roms == np.min(c_roms)] = 1


In [ ]:
yi_roms_min, xi_roms_min = np.min(np.where(used_area_roms == 1), axis=1)
yi_roms_max, xi_roms_max = np.max(np.where(used_area_roms == 1), axis=1)
print(yi_roms_min, xi_roms_min)
print(yi_roms_max, xi_roms_max)

In [ ]:
yi_roms_min, xi_roms_min, yi_roms_max, xi_roms_max = 444, 84, 638, 222


In [ ]:
# add buffer for sea ice drift near edge
yi_roms_min -= 50
xi_roms_min -= 50
yi_roms_max += 50
xi_roms_max += 50
plt.imshow(example_roms_surface_m00['ice_concentration'].isel(time=0).isel(X=slice(xi_roms_min+1,xi_roms_max+1), Y=slice(yi_roms_min,yi_roms_max)))


### Simulate sea ice trajectories for each complete forecast

Missing file 19/04 -> skip in run (adapt for j and ds_roms line)

In [ ]:
model_name = 'MET-ROMS_m00'
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)
output_subfolder = os.path.join(output_folder, 'trajs_alt')
os.makedirs(output_subfolder, exist_ok=True) 

start_dt = datetime(2024,4,1)
reader_landmask = reader_global_landmask.Reader()

# Loop through dates
for j in range(0, 18):
#for j in range(19, 61):
    file_date = start_dt + timedelta(days=j)
    o = SeaIceDrift(loglevel=0)
    file_date_str = file_date.strftime('%Y%m%d%H')
    ds_roms = xr.open_dataset(file_list_roms_m00[j])
    #ds_roms = xr.open_dataset(file_list_roms_m00[j-1])
    i = 0

    reader_roms = reader_netCDF_CF_generic.Reader(ds_roms, \
        proj4='+proj=lcc +lat_0=77.5 +lon_0=-25 +lat_1=77.5 +lat_2=77.5 +no_defs +R=6.371e+06')

    o.add_reader([reader_landmask, reader_roms])
    for tr_nr in [x for x in range(1, 36) if x not in [5, 33]]:
        lon0 = dr[tr_nr]['lon'].interp(time=file_date, method = 'nearest')
        lat0 = dr[tr_nr]['lat'].interp(time=file_date, method = 'nearest')
        time_diff = np.abs((dr[tr_nr]['lon']['time']- np.datetime64(file_date)) / np.timedelta64(1, 'h'))
        min_time_diff = np.min(time_diff).values

        if (min_time_diff<0.15) and not (np.isnan(lon0) or np.isnan(lat0)):

            o.seed_elements(lon=lon0, lat=lat0, time=file_date, origin_marker=tr_nr)
            print(f"{tr_nr} seeded")
            
            i = 1
            
    if i != 0:
        # run with 30 minute resolution up to end time of the forecast
        o.run(end_time=reader_roms.end_time, time_step=1800, time_step_output=1800, outfile=os.path.join(output_subfolder, f"trajs_{model_name}_{file_date_str}.nc"))


In [ ]:
model_name = 'MET-ROMS_m01'
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)
output_subfolder = os.path.join(output_folder, 'trajs_alt')
os.makedirs(output_subfolder, exist_ok=True) 

start_dt = datetime(2024,4,1)
reader_landmask = reader_global_landmask.Reader()

# Loop through dates
for j in range(0, 18):
#for j in range(19, 61):
    file_date = start_dt + timedelta(days=j)
    o = SeaIceDrift(loglevel=0)
    file_date_str = file_date.strftime('%Y%m%d%H')
    ds_roms = xr.open_dataset(file_list_roms_m01[j])
    #ds_roms = xr.open_dataset(file_list_roms_m01[j-1])
    i = 0
 
    reader_roms = reader_netCDF_CF_generic.Reader(ds_roms, \
        proj4='+proj=lcc +lat_0=77.5 +lon_0=-25 +lat_1=77.5 +lat_2=77.5 +no_defs +R=6.371e+06')
    o.add_reader([reader_landmask, reader_roms])
    for tr_nr in [x for x in range(1, 36) if x not in [5, 33]]:
        lon0 = dr[tr_nr]['lon'].interp(time=file_date, method = 'nearest')
        lat0 = dr[tr_nr]['lat'].interp(time=file_date, method = 'nearest')
        time_diff = np.abs((dr[tr_nr]['lon']['time']- np.datetime64(file_date)) / np.timedelta64(1, 'h'))
        min_time_diff = np.min(time_diff).values

        if (min_time_diff<0.15) and not (np.isnan(lon0) or np.isnan(lat0)):

            o.seed_elements(lon=lon0, lat=lat0, time=file_date, origin_marker=tr_nr)
            print(f"{tr_nr} seeded")
            
            i = 1
            
    if i != 0:
        # run with 30 minute resolution up to end time of the forecast
        o.run(end_time=reader_roms.end_time, time_step=1800, time_step_output=1800, outfile=os.path.join(output_subfolder, f"trajs_{model_name}_{file_date_str}.nc"))


Plot trajectory simulation to compare later with reconstruction

In [ ]:
start_dt = datetime(2024,4,16)
model_name = 'MET-ROMS_m00'
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)
output_subfolder = os.path.join(output_folder, 'trajs_alt')
os.makedirs(output_subfolder, exist_ok=True) 
file_date_str = start_dt.strftime('%Y%m%d00')
a=xr.open_dataset(os.path.join(output_subfolder, f"trajs_{model_name}_{file_date_str}.nc"))
a['lon'].sel(trajectory=3).plot()

In [ ]:
start_dt = datetime(2024,4,16)
model_name = 'MET-ROMS_m01'
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)
output_subfolder = os.path.join(output_folder, 'trajs_alt')
os.makedirs(output_subfolder, exist_ok=True) 
file_date_str = start_dt.strftime('%Y%m%d00')
a=xr.open_dataset(os.path.join(output_subfolder, f"trajs_{model_name}_{file_date_str}.nc"))
a['lon'].sel(trajectory=3).plot()

In [ ]:
a

### Fill in overarching dataset (m00)

In [ ]:
# Initialize variables and paths
model_name = 'MET-ROMS_m00'
#model_name = 'MET-ROMS_m01'
example_model_ds = xr.open_dataset(f'example_{model_name}.nc',engine="netcdf4")
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)
output_subfolder = os.path.join(output_folder, 'trajs_alt')
os.makedirs(output_subfolder, exist_ok=True)

key_lon, key_lat, key_x, key_y = 'lon', 'lat', 'X', 'Y'
alpha_ds = xr.open_dataset('alpha_MET-ROMS.nc')

In [ ]:
# Calculate yx_model in parallel
yx_model = {}
with ThreadPoolExecutor() as executor:
    futures = {executor.submit(calculate_tr_nr_yx_model, example_model_ds, key_lon, key_lat, key_x, key_y, dataset, tr_nr): tr_nr for tr_nr in tr_nr_values}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Calculating yx_model"):
        tr_nr, y_model, x_model = future.result()
        yx_model[tr_nr] = (y_model, x_model)
        
with open(f'yx_model_MET-ROMS.pkl', 'wb') as f:
    pickle.dump(yx_model,f)

In [ ]:
with open(f'yx_model_MET-ROMS.pkl', 'rb') as f:
    yx_model = pickle.load(f)

In [ ]:
for file_date in file_dates:
    try:
        path_roms = "/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/MET-ROMS/archive/surface"
        file_date_str_p = file_date.strftime('%Y/%m/%d/T%HZ')
        file_date_str_np = file_date.strftime('%Y%m%dT%HZ')
        #print(f"{path_roms}/{file_date_str_p}/barents_sfc_{file_date_str_np}m00.nc")
        xr.open_dataset(f"{path_roms}/{file_date_str_p}/barents_sfc_{file_date_str_np}m01.nc")
    except:
        print(f'file not found {file_date_str_p}')

### Fill in overarching dataset with forecast variables

In [ ]:
# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

def process_forecasts(file_date, output_folder, model_name, key_lon, key_lat, dataset, yx_model):
    if file_date != datetime(2024, 4, 19, 0, 0):
        path_roms = "/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/MET-ROMS/archive/surface"
        file_date_str_p = file_date.strftime('%Y/%m/%d/T%HZ')
        file_date_str_np = file_date.strftime('%Y%m%dT%HZ')
        logging.info(f"Processing forecasts for date: {file_date_str_p}")

        alpha_rad = alpha_ds['__xarray_dataarray_variable__'] * np.pi / 180

        # Try opening forecasts
        try:
            ds_fc = xr.open_dataset(f"{path_roms}/{file_date_str_p}/barents_sfc_{file_date_str_np}m00.nc")
            #ds_fc = xr.open_dataset(f"{path_roms}/{file_date_str_p}/barents_sfc_{file_date_str_np}m01.nc", engine='netcdf4')
            
            ds_fc['Uice'] = ds_fc['ice_u'] * np.cos(alpha_rad) - ds_fc['ice_v'] * np.sin(alpha_rad)
            ds_fc['Vice'] = ds_fc['ice_u'] * np.sin(alpha_rad) + ds_fc['ice_v'] * np.cos(alpha_rad)


            #ds_fc['Uice'] = ds_fc['ice_u']
            #ds_fc['Vice'] = ds_fc['ice_v']

            updated_dataset = dataset.copy()
            time_ds = dataset.time_ds

            for tr_nr, (y_model, x_model) in yx_model.items():
                mask_nan = ~(np.isnan(x_model) + np.isnan(y_model))
                for i in range(0, 240, 24):
                    lt_int = f']{i}_{i+24}]'
                    left_lt = ((ds_fc.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))> i+0.1
                    right_lt = ((ds_fc.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))<= i+24+0.1
                    sel_lt = np.where(left_lt*right_lt)[0]
                    ds_fc_lt_int = ds_fc.isel(time=sel_lt)
                    av_times_fc = ds_fc_lt_int['time'].values
                    mask_av = pd.to_datetime(time_ds).isin(pd.to_datetime(av_times_fc))

                    if np.sum(mask_av * mask_nan) > 0:
                        for k in range(0,4):
                            var_name_riops = ['ice_concentration', 'Uice', 'Vice', 'ice_thickness'][k]
                            var_name_ds = ['SIC', 'Uice', 'Vice', 'Hice'][k]
                            var_model_lt_int_path = ds_fc[var_name_riops].sel({'time': time_ds[mask_av * mask_nan], key_x: x_model[mask_av * mask_nan], key_y: y_model[mask_av * mask_nan]})
                            updated_dataset[var_name_ds].loc[dict(model=model_name, tr_nr=tr_nr, lt_int=lt_int, time_ds=time_ds[mask_av * mask_nan])] = var_model_lt_int_path.values
            return updated_dataset

        except FileNotFoundError:
            logging.warning(f"File not found: {path_roms}/{file_date_str_p}/barents_sfc_{file_date_str_np}m00.nc")
            return dataset
    else:
        return dataset
        

    

# Process forecasts in parallel
for file_date in file_dates:
    dataset = process_forecasts(file_date, output_folder, model_name, key_lon, key_lat, dataset, yx_model)
    
# Process forecasts in parallel
#with ThreadPoolExecutor() as executor:
#    futures = [
#        executor.submit(partial(process_forecasts, file_date, output_folder, model_name, key_lon, key_lat, dataset, yx_model))
#        for file_date in tqdm(file_dates, desc="Processing forecasts")
#    ]
#    for future in tqdm(futures, desc="Updating dataset with forecast results"):
#        result = future.result()
#        if result is not None:
#            dataset.update(result)


### Fill in overarching dataset with simulated drift trajectories

In [ ]:
def process_trajectories(file_date, output_subfolder, model_name, dataset):
    if file_date != datetime(2024, 4, 19, 0, 0):
        file_date_str = file_date.strftime('%Y%m%d%H')
        logging.info(f"Processing trajectories for date: {file_date_str}")

        file_name_traj = os.path.join(output_subfolder, f"trajs_{model_name}_{file_date_str}.nc")
        try:
            ds_traj = xr.open_dataset(file_name_traj)

            updated_dataset = dataset.copy()
            trajectory_sim = 0

            for tr_nr in np.unique(ds_traj['origin_marker'].values):
                if not np.isnan(tr_nr):
                    trajectory_sim += 1
                    for i in range(0, 240, 24):
                        lt_int = f']{i}_{i + 24}]'
                        left_lt = ((ds_traj.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))> i+0.1
                        right_lt = ((ds_traj.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))<= i+24+0.1
                        sel_lt = np.where(left_lt*right_lt)[0]                
                        time_traj = ds_traj.sel(trajectory=trajectory_sim)['time'].isel(time=sel_lt)
                        time_traj = time_traj.sel(time = (time_traj.time <= np.datetime64('2024-06-01T00:00')))
                        if len(time_traj.values)>0:
                            updated_dataset['lon_sim'].loc[dict(model=model_name, tr_nr=tr_nr, lt_int=lt_int, time_ds=time_traj)] = ds_traj['lon'].sel(trajectory=trajectory_sim, time=time_traj)
                            updated_dataset['lat_sim'].loc[dict(model=model_name, tr_nr=tr_nr, lt_int=lt_int, time_ds=time_traj)] = ds_traj['lat'].sel(trajectory=trajectory_sim, time=time_traj)

            return updated_dataset

        except FileNotFoundError:
            logging.warning(f"File not found: {file_name_traj}")
            return dataset
    else:
        return dataset     
    

    


# Process trajectories in parallel
with ThreadPoolExecutor() as executor:
    futures = [
        executor.submit(partial(process_trajectories, file_date, output_subfolder, model_name, dataset))
        for file_date in tqdm(file_dates, desc="Processing trajectories")
    ]
    for future in tqdm(futures, desc="Updating dataset with trajectory results"):
        result = future.result()
        if result is not None:
            dataset.update(result)


### Fill in overarching dataset (m01)

In [ ]:
# Initialize variables and paths
#model_name = 'MET-ROMS_m00'
model_name = 'MET-ROMS_m01'
example_model_ds = xr.open_dataset(f'example_{model_name}.nc',engine="netcdf4")
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)
output_subfolder = os.path.join(output_folder, 'trajs_alt')
os.makedirs(output_subfolder, exist_ok=True)

key_lon, key_lat, key_x, key_y = 'lon', 'lat', 'X', 'Y'
alpha_ds = xr.open_dataset('alpha_MET-ROMS.nc')

### Fill in overarching dataset with forecast variables

In [ ]:
# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

def process_forecasts(file_date, output_folder, model_name, key_lon, key_lat, dataset, yx_model):
    if file_date != datetime(2024, 4, 19, 0, 0):
        path_roms = "/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/MET-ROMS/archive/surface"
        file_date_str_p = file_date.strftime('%Y/%m/%d/T%HZ')
        file_date_str_np = file_date.strftime('%Y%m%dT%HZ')
        logging.info(f"Processing forecasts for date: {file_date_str_p}")

        alpha_rad = alpha_ds['__xarray_dataarray_variable__'] * np.pi / 180

        # Try opening forecasts
        try:
            #ds_fc = xr.open_dataset(f"{path_roms}/{file_date_str_p}/barents_sfc_{file_date_str_np}m00.nc")
            ds_fc = xr.open_dataset(f"{path_roms}/{file_date_str_p}/barents_sfc_{file_date_str_np}m01.nc", engine='netcdf4')
            
            ds_fc['Uice'] = ds_fc['ice_u'] * np.cos(alpha_rad) - ds_fc['ice_v'] * np.sin(alpha_rad)
            ds_fc['Vice'] = ds_fc['ice_u'] * np.sin(alpha_rad) + ds_fc['ice_v'] * np.cos(alpha_rad)


            #ds_fc['Uice'] = ds_fc['ice_u']
            #ds_fc['Vice'] = ds_fc['ice_v']

            updated_dataset = dataset.copy()
            time_ds = dataset.time_ds

            for tr_nr, (y_model, x_model) in yx_model.items():
                mask_nan = ~(np.isnan(x_model) + np.isnan(y_model))
                for i in range(0, 240, 24):
                    lt_int = f']{i}_{i+24}]'
                    left_lt = ((ds_fc.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))> i+0.1
                    right_lt = ((ds_fc.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))<= i+24+0.1
                    sel_lt = np.where(left_lt*right_lt)[0]
                    ds_fc_lt_int = ds_fc.isel(time=sel_lt)
                    av_times_fc = ds_fc_lt_int['time'].values
                    mask_av = pd.to_datetime(time_ds).isin(pd.to_datetime(av_times_fc))

                    if np.sum(mask_av * mask_nan) > 0:
                        for k in range(0,4):
                            var_name_riops = ['ice_concentration', 'Uice', 'Vice', 'ice_thickness'][k]
                            var_name_ds = ['SIC', 'Uice', 'Vice', 'Hice'][k]
                            var_model_lt_int_path = ds_fc[var_name_riops].sel({'time': time_ds[mask_av * mask_nan], key_x: x_model[mask_av * mask_nan], key_y: y_model[mask_av * mask_nan]})
                            updated_dataset[var_name_ds].loc[dict(model=model_name, tr_nr=tr_nr, lt_int=lt_int, time_ds=time_ds[mask_av * mask_nan])] = var_model_lt_int_path.values
            ds_fc.close()
            return updated_dataset

        except FileNotFoundError:
            logging.warning(f"File not found: {path_roms}/{file_date_str_p}/barents_sfc_{file_date_str_np}m00.nc")
            return dataset
    else:
        return dataset
        

    

# Process forecasts in parallel
for file_date in file_dates[59:]:
    dataset = process_forecasts(file_date, output_folder, model_name, key_lon, key_lat, dataset, yx_model)
    
# Process forecasts in parallel
#with ThreadPoolExecutor() as executor:
#    futures = [
#        executor.submit(partial(process_forecasts, file_date, output_folder, model_name, key_lon, key_lat, dataset, yx_model))
#        for file_date in tqdm(file_dates, desc="Processing forecasts")
#    ]
#    for future in tqdm(futures, desc="Updating dataset with forecast results"):
#        result = future.result()
#        if result is not None:
#            dataset.update(result)


### Fill in overarching dataset with simulated drift trajectories

In [ ]:
def process_trajectories(file_date, output_subfolder, model_name, dataset):
    if file_date != datetime(2024, 4, 19, 0, 0):
        file_date_str = file_date.strftime('%Y%m%d%H')
        logging.info(f"Processing trajectories for date: {file_date_str}")

        file_name_traj = os.path.join(output_subfolder, f"trajs_{model_name}_{file_date_str}.nc")
        try:
            ds_traj = xr.open_dataset(file_name_traj)

            updated_dataset = dataset.copy()
            trajectory_sim = 0

            for tr_nr in np.unique(ds_traj['origin_marker'].values):
                if not np.isnan(tr_nr):
                    trajectory_sim += 1
                    for i in range(0, 240, 24):
                        lt_int = f']{i}_{i + 24}]'
                        left_lt = ((ds_traj.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))> i+0.1
                        right_lt = ((ds_traj.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))<= i+24+0.1
                        sel_lt = np.where(left_lt*right_lt)[0]                
                        time_traj = ds_traj.sel(trajectory=trajectory_sim)['time'].isel(time=sel_lt)
                        time_traj = time_traj.sel(time = (time_traj.time <= np.datetime64('2024-06-01T00:00')))
                        if len(time_traj.values)>0:
                            updated_dataset['lon_sim'].loc[dict(model=model_name, tr_nr=tr_nr, lt_int=lt_int, time_ds=time_traj)] = ds_traj['lon'].sel(trajectory=trajectory_sim, time=time_traj)
                            updated_dataset['lat_sim'].loc[dict(model=model_name, tr_nr=tr_nr, lt_int=lt_int, time_ds=time_traj)] = ds_traj['lat'].sel(trajectory=trajectory_sim, time=time_traj)

            return updated_dataset

        except FileNotFoundError:
            logging.warning(f"File not found: {file_name_traj}")
            return dataset
    else:
        return dataset     
    

    


# Process trajectories in parallel
with ThreadPoolExecutor() as executor:
    futures = [
        executor.submit(partial(process_trajectories, file_date, output_subfolder, model_name, dataset))
        for file_date in tqdm(file_dates, desc="Processing trajectories")
    ]
    for future in tqdm(futures, desc="Updating dataset with trajectory results"):
        result = future.result()
        if result is not None:
            dataset.update(result)


In [ ]:
dataset.to_netcdf('traj_ds_roms_no_vars.nc')

### Plot to check data

In [ ]:
buoy_nr = 3
buoy_data = dataset.sel(tr_nr = buoy_nr)

colors = {
    'NERSC-NextSIM': 'tab:blue',
    'ECCC-RIOPS': 'tab:orange',
    'ECCC-RIOPScoupled': 'tab:green',
    'NRL-GOFS': 'tab:red',
    'MET-TOPAZ5': 'tab:purple',
    'MET-ROMS_m00': 'tab:cyan',
    'MET-ROMS_m01': 'tab:pink'
}

#['NERSC-NextSIM', 'ECCC-RIOPS', 'ECCC-RIOPScoupled', 'NRL-GOFS','MET-TOPAZ5', 'MET-ROMS_m00', 'MET-ROMS_m01']
#models_sel = []
models_sel = ['MET-ROMS_m00', 'MET-ROMS_m01']
target_date = '2024-04-16'  # Example date in YYYY-MM-DD format

lon_obs_b, lat_obs_b = buoy_data['lon_obs'], buoy_data['lat_obs']
plt.figure()
start = pd.to_datetime(target_date) + pd.DateOffset(seconds=1)
end = start + pd.DateOffset(days=4)
        
lon_obs_b.sel(time_ds=slice(start,end)).plot(color='k')
for model_name in models_sel:
    color = colors.get(model_name, 'black')
    
    # Plot dummy line for legend
    plt.plot([], [], color=color, label=model_name)
    
    for k in range(0,4):
        lt_int_sel = dataset['lt_int'].values[k]
        start = pd.to_datetime(target_date) + pd.DateOffset(seconds=1) + pd.DateOffset(days=k) 
        end = start + pd.DateOffset(days=1)
        lon_sim_lt_k = buoy_data['lon_sim'].sel(model=model_name, time_ds=slice(start,end), lt_int=lt_int_sel)
        lat_sim_lt_k = buoy_data['lat_sim'].sel(model=model_name, time_ds=slice(start,end), lt_int=lt_int_sel)
        lon_sim_lt_k.plot(color=color)

plt.legend(loc='upper left')
plt.grid(True)

In [ ]:
tr_nr = 2
dataset['lat_obs'].sel(tr_nr=tr_nr).plot()
dataset['lat_sim'].sel(tr_nr=tr_nr, model='MET-ROMS_m00', lt_int=']0_24]').dropna(dim='time_ds', how='any').plot()
dataset['lat_sim'].sel(tr_nr=tr_nr, model='MET-ROMS_m01', lt_int=']0_24]').dropna(dim='time_ds', how='any').plot()

## MET-LAON

In [ ]:
model_name = 'MET-LAON'

In [ ]:
# MET-LAON
# Coordinates: time,nj,ni
# Variables: uvel_h (ice velocity (x) in m/s), vvel_h (ice velocity (y) in m/s), aice_h (ice area  (aggregate)),
# hi_h (grid cell mean ice thickness in m), hi_s (grid cell mean snow thickness in m)
# time: 144 time stamps per file ROUGHLY hourly, starting 1h after initialization until +7d
example_met_laon = xr.open_dataset('/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/MET-LAON/iceh_01h.2024-04-01.nc',engine="netcdf4")


In [ ]:
example_met_laon

In [ ]:
example_met_laon['time']

In [ ]:
example_met_laon['time'] = np.array(example_met_laon['time'] + np.timedelta64(1, 'm'), dtype='datetime64[h]')

In [ ]:
example_met_laon['time']

### Create list of files of interest

In [ ]:
path_met_laon= "/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/MET-LAON/"
file_list_met_laon = list(glob(path_met_laon+ "*.nc"))
file_list_met_laon.sort()
file_list_met_laon

### Determine relevant grid area + buffer zone

In [ ]:
used_area_met_laon_T = np.zeros(example_met_laon['TLAT'].values.shape)

for tr_nr in [x for x in range(1, 36) if x not in [5, 33]]:    
    # loop through each position, indicate closest point in used_area_topaz5
    for time in dr[tr_nr]['time']:
        lat_time = dr[tr_nr]['lat'].sel(time=time).values
        lon_time = dr[tr_nr]['lon'].sel(time=time).values
        
        c_met_laon = np.maximum(np.abs(example_met_laon['TLAT'].values-lat_time), np.abs(example_met_laon['TLON'].values-lon_time))
        used_area_met_laon_T[c_met_laon == np.min(c_met_laon)] = 1


In [ ]:
used_area_met_laon_U = np.zeros(example_met_laon['ULAT'].values.shape)

for tr_nr in [x for x in range(1, 36) if x not in [5, 33]]:    
    # loop through each position, indicate closest point in used_area_topaz5
    for time in dr[tr_nr]['time']:
        lat_time = dr[tr_nr]['lat'].sel(time=time).values
        lon_time = dr[tr_nr]['lon'].sel(time=time).values
        
        c_met_laon = np.maximum(np.abs(example_met_laon['ULAT'].values-lat_time), np.abs(example_met_laon['ULON'].values-lon_time))
        used_area_met_laon_U[c_met_laon == np.min(c_met_laon)] = 1


DOES NOT WORK! Gives error after 22/04, simulations are off (appr constant, only small variations)

Possibly due to grid structure and absence of proj4 string

In [ ]:
model_name = 'MET-LAON'
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)
output_subfolder = os.path.join(output_folder, 'trajs_alt')
os.makedirs(output_subfolder, exist_ok=True) 

start_dt = datetime(2024,4,1,1,0,0)
reader_landmask = reader_global_landmask.Reader()

# Loop through dates
for j in range(7, 55):
    file_date = start_dt + timedelta(days=j)
    print(file_date)
    o = SeaIceDrift(loglevel=0)
    file_date_str = file_date.strftime('%Y%m%d%H')
    ds_laon = xr.open_dataset(file_list_met_laon[j])
    ds_laon['time'] = np.array(ds_laon['time'] + np.timedelta64(1, 'm'), dtype='datetime64[h]').astype('datetime64[ns]')
    print(ds_laon['time'])
    ds_laon['ULAT'].attrs['standard_name'] = 'latitude'
    ds_laon['ULON'].attrs['standard_name'] = 'longitude'
    ds_laon['uvel_h'].attrs['standard_name'] = 'sea_ice_x_velocity'
    ds_laon['vvel_h'].attrs['standard_name'] = 'sea_ice_y_velocity'
    ds_laon['time'].attrs['standard_name'] = 'time'
    i = 0

    reader_laon = reader_netCDF_CF_generic.Reader(ds_laon)

    o.add_reader([reader_landmask, reader_laon])
    for tr_nr in [x for x in range(1, 36) if x not in [5, 33]]:
        lon0 = dr[tr_nr]['lon'].interp(time=file_date, method = 'nearest')
        lat0 = dr[tr_nr]['lat'].interp(time=file_date, method = 'nearest')
        time_diff = np.abs((dr[tr_nr]['lon']['time']- np.datetime64(file_date)) / np.timedelta64(1, 'h'))
        min_time_diff = np.min(time_diff).values

        if (min_time_diff<0.15) and not (np.isnan(lon0) or np.isnan(lat0)):

            o.seed_elements(lon=lon0, lat=lat0, time=file_date, origin_marker=tr_nr)
            print(f"{tr_nr} seeded")
            
            i = 1
            
    if i != 0:
        # run with 30 minute resolution up to end time of the forecast
        o.run(end_time=reader_laon.end_time, time_step=1800, time_step_output=1800, outfile=os.path.join(output_subfolder, f"trajs_{model_name}_{file_date_str}.nc"))


In [ ]:
start_dt = datetime(2024,4,16)
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)
output_subfolder = os.path.join(output_folder, 'trajs_alt')
os.makedirs(output_subfolder, exist_ok=True) 
file_date_str = start_dt.strftime('%Y%m%d01')
a=xr.open_dataset(os.path.join(output_subfolder, f"trajs_{model_name}_{file_date_str}.nc"))
a['lon'].sel(trajectory=3).plot()

In [ ]:
a

TO BE COMPLETED

## DMI-HYCOM

In [ ]:
model_name = 'DMI-HYCOM'

In [ ]:
# DMI-HYCOM
# Coordinates: time, ni, nj
# Variables: uvel (ice velocity (x) in m/s), vvel (ice velocity (y) in m/s), aice (ice area  (aggregate)),
# hi (grid cell mean ice thickness in m), hi_s (grid cell mean snow thickness in m)
# time: 1 time stamp per file
example_dmi_hycom = xr.open_dataset('/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/DMI-HYCOM/2024042100/Svalbardreduced.2024-04-25-82800.nc',engine="netcdf4")


In [ ]:
example_dmi_hycom.to_netcdf(f'example_{model_name}.nc')

In [ ]:
example_dmi_hycom = xr.open_dataset('example_DMI-HYCOM.nc')

In [ ]:
example_dmi_hycom 

In [ ]:
example_dmi_hycom['time']

### Calculate local orientation of grid to rotate sea ice drift vectors

In [ ]:
# Variables 
xx = example_dmi_hycom.ni
yy = example_dmi_hycom.nj
lat = example_dmi_hycom.ULAT
lon = example_dmi_hycom.ULON
alphaa=lat #Target matrix

for j in range(0,yy.size-1):
    for i in range(0,xx.size-1):
        # Prevent out of bounds
        if j==yy.size-1:
            j1=j-1; j2=j
        else:
            j1=j; j2=j+1
        if i==xx.size-1:
            i1=i-1; i2=i
        else:
            i1=i; i2=i+1

        dlatykm=distance([lat[j1,i1],lon[j1,i1]],[lat[j2,i1],lon[j1,i1]])
        dlonykm=distance([lat[j1,i1],lon[j1,i1]],[lat[j1,i1],lon[j2,i1]])
        
        alphaa[j,i]=np.arctan2(dlatykm,dlonykm)*180/np.pi - 90
alphaa_ds = xr.DataArray(alphaa, dims=['nj', 'ni'])
alphaa_ds.to_netcdf('alpha_'+model_name+'_U.nc')

### Create list of files of interest

In [ ]:
path_dmi_hycom = "/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/DMI-HYCOM/"
# some md5checksum and temp files as well => only select daily nc files
file_list_dmi_hycom = list(glob(path_dmi_hycom + "*/*.nc"))

file_list_dmi_hycom.sort()


In [ ]:
file_list_dmi_hycom

### Determine relevant grid area + buffer zone

In [ ]:
used_area_dmi_hycom = np.zeros(example_dmi_hycom['ULAT'].values.shape)

for tr_nr in [x for x in range(1, 36) if x not in [5, 33]]:    
    # loop through each position, indicate closest point in used_area_topaz5
    for time in dr[tr_nr]['time']:
        lat_time = dr[tr_nr]['lat'].sel(time=time).values
        lon_time = dr[tr_nr]['lon'].sel(time=time).values
        
        c_dmi_hycom = np.maximum(np.abs(example_dmi_hycom['ULAT'].values-lat_time), np.abs(example_dmi_hycom['ULON'].values-lon_time))
        used_area_dmi_hycom[c_dmi_hycom == np.min(c_dmi_hycom)] = 1


In [ ]:
yi_dmi_hycom_min, xi_dmi_hycom_min = np.min(np.where(used_area_dmi_hycom == 1), axis=1)
yi_dmi_hycom_max, xi_dmi_hycom_max = np.max(np.where(used_area_dmi_hycom == 1), axis=1)
print(yi_dmi_hycom_min, xi_dmi_hycom_min)
print(yi_dmi_hycom_max, xi_dmi_hycom_max)

In [ ]:
yi_dmi_hycom_min, xi_dmi_hycom_min, yi_dmi_hycom_max, xi_dmi_hycom_max = 307, 179, 416, 247

In [ ]:
# add buffer for sea ice drift near edge
yi_dmi_hycom_min -= 50
xi_dmi_hycom_min -= 50
yi_dmi_hycom_max += 50
xi_dmi_hycom_max += 50
plt.imshow(example_dmi_hycom['uvel'].isel(time=0).isel(ni=slice(xi_dmi_hycom_min,xi_dmi_hycom_max+1), nj=slice(yi_dmi_hycom_min,yi_dmi_hycom_max+1)))


### Concatenate forecasts files to one file per initialisation date, only selecting relevant area and buffer zone
### Rename variables to reference names

In [ ]:
import os
from datetime import datetime, timedelta
from functools import partial
from glob import glob
import numpy as np
import xarray as xr
from concurrent.futures import ProcessPoolExecutor

output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)

start_dt = datetime(2024, 4, 1)

def preprocess_file(file, x_bnds, y_bnds):
    x = xr.open_dataset(file).load()
    ds = xr.Dataset()
    ds['SIC'] = x['aice'].isel(ni=slice(*x_bnds), nj=slice(*y_bnds))
    ds['Hice'] = x['hi'].isel(ni=slice(*x_bnds), nj=slice(*y_bnds))
    ds['sea_ice_x_velocity'] = x['uvel'].isel(ni=slice(*x_bnds), nj=slice(*y_bnds))
    ds['sea_ice_y_velocity'] = x['vvel'].isel(ni=slice(*x_bnds), nj=slice(*y_bnds))

    base_name = os.path.basename(file)
    date_str, seconds_str = base_name.split('.')[1].rsplit('-', 1)
    file_time = datetime.strptime(date_str, '%Y-%m-%d') + timedelta(seconds=int(seconds_str))
    ds['time'] = xr.DataArray(np.array([file_time]), dims='time', coords={'time': [file_time]})
    ds = ds.assign_coords(time=ds['time'])
    return ds

x_bnds = (xi_dmi_hycom_min, xi_dmi_hycom_max + 1)
y_bnds = (yi_dmi_hycom_min, yi_dmi_hycom_max + 1)

for date in [start_dt + timedelta(days=i) for i in range(0, 61)]:
    date_str = date.strftime('%Y%m%d')
    if date_str not in ['20240427', '20240428']:
        print(date_str)
        file_list = sorted(glob(f"{path_dmi_hycom}/{date_str}00/*.nc"))
        with ProcessPoolExecutor() as executor:
            datasets = list(executor.map(partial(preprocess_file, x_bnds=x_bnds, y_bnds=y_bnds), file_list))

        dmi_hycom_ds_date = xr.concat(datasets, dim='time')
        dmi_hycom_ds_date.to_netcdf(os.path.join(output_folder, f"{model_name}_{date_str}.nc"))


In [ ]:
a =xr.open_dataset('DMI-HYCOM_conc_forecasts/DMI-HYCOM_20240402.nc')
a

In [ ]:
a['sea_ice_y_velocity'].isel(nj=100, ni=80).plot()

In [ ]:
a['sea_ice_x_velocity'].isel(nj=100, ni=80).plot()

### Simulate sea ice trajectories for each complete forecast

Simulations are off, possibly due to grid structure and absence of projection string

In [ ]:
reader_dmi_hycom

In [ ]:
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)
output_subfolder = os.path.join(output_folder, 'trajs_alt')
os.makedirs(output_subfolder, exist_ok=True) 
# forecast starts at 01:00
start_dt = datetime(2024,4,1,1)
reader_landmask = reader_global_landmask.Reader()

# Loop through dates
for file_date in [start_dt + timedelta(hours=i) for i in range(0, 61*24,24)]:
    i = 0
    o = SeaIceDrift(loglevel=0)
    file_date_str = file_date.strftime('%Y%m%d')
    if file_date_str not in ['20240427', '20240428']:
        ds_dmi_hycom = xr.open_dataset(os.path.join(output_folder, f"{model_name}_{file_date_str}.nc"))
        ds_dmi_hycom['sea_ice_x_velocity'].attrs['standard_name'] = 'sea_ice_x_velocity'
        ds_dmi_hycom['sea_ice_y_velocity'].attrs['standard_name'] = 'sea_ice_y_velocity'
        ds_dmi_hycom['sea_ice_x_velocity'].attrs['units'] = 'm s-1'
        ds_dmi_hycom['sea_ice_y_velocity'].attrs['units'] = 'm s-1'
            
        
        ds_dmi_hycom['sea_ice_x_velocity'] = ds_dmi_hycom['sea_ice_x_velocity'].assign_coords(
    lon=(('nj','ni'), ds_dmi_hycom['ULON'].data),
    lat=(('nj','ni'), ds_dmi_hycom['ULAT'].data)
)
        ds_dmi_hycom['sea_ice_y_velocity'] = ds_dmi_hycom['sea_ice_y_velocity'].assign_coords(
            lon=(('nj','ni'), ds_dmi_hycom['ULON'].data),
            lat=(('nj','ni'), ds_dmi_hycom['ULAT'].data)
        )

        ds_dmi_hycom['sea_ice_x_velocity'].attrs['coordinates'] = 'ULON ULAT'
        ds_dmi_hycom['sea_ice_y_velocity'].attrs['coordinates'] = 'ULON ULAT'


        reader_dmi_hycom = reader_netCDF_CF_generic.Reader(ds_dmi_hycom)
        #print(reader_dmi_hycom)
        o.add_reader([reader_landmask, reader_dmi_hycom])
        for tr_nr in [x for x in range(1, 36) if x not in [5, 33]]:

            lon0 = dr[tr_nr]['lon'].interp(time=file_date, method = 'nearest')
            lat0 = dr[tr_nr]['lat'].interp(time=file_date, method = 'nearest')
            time_diff = np.abs((dr[tr_nr]['lon']['time']- np.datetime64(file_date)) / np.timedelta64(1, 'h'))
            min_time_diff = np.min(time_diff).values

            if (min_time_diff<0.15) and not (np.isnan(lon0) or np.isnan(lat0)):
                print(file_date)
                o.seed_elements(lon=lon0, lat=lat0, time=file_date, origin_marker=tr_nr)
                print(f"{tr_nr} seeded")

                i = 1

        if i != 0:
            # run with 30 minute resolution up to end time of the forecast
            o.run(end_time=reader_dmi_hycom.end_time, time_step=1800, time_step_output=1800, outfile=os.path.join(output_subfolder, f"trajs_{model_name}_{file_date_str}.nc"))


Plot trajectory simulation to compare with reconstruction

In [ ]:
start_dt = datetime(2024,4,16)
model_name = 'DMI-HYCOM'
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)
output_subfolder = os.path.join(output_folder, 'trajs_alt')
os.makedirs(output_subfolder, exist_ok=True) 
file_date_str = start_dt.strftime('%Y%m%d')
a=xr.open_dataset(os.path.join(output_subfolder, f"trajs_{model_name}_{file_date_str}.nc"))
a['lon'].sel(trajectory=3).plot()

In [ ]:
a

### Fill in overarching dataset

In [ ]:
# Initialize variables and paths
model_name = 'DMI-HYCOM'
example_model_ds = xr.open_dataset(f'example_{model_name}.nc',engine="netcdf4")
output_folder = model_name + '_conc_forecasts'
os.makedirs(output_folder, exist_ok=True)
output_subfolder = os.path.join(output_folder, 'trajs_alt')
os.makedirs(output_subfolder, exist_ok=True)

key_lon_U, key_lat_U,key_lon_T, key_lat_T, key_x, key_y = 'ULON', 'ULAT','TLON', 'TLAT', 'ni', 'nj'
alpha_ds_U = xr.open_dataset('alpha_DMI-HYCOM_U.nc')

Two different grids -> two different yx_models!

In [ ]:
# Calculate yx_model in parallel
yx_model = {}
with ThreadPoolExecutor() as executor:
    futures = {executor.submit(calculate_tr_nr_yx_model, example_model_ds, key_lon_U, key_lat_U, key_x, key_y, dataset, tr_nr): tr_nr for tr_nr in tr_nr_values}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Calculating yx_model"):
        tr_nr, y_model, x_model = future.result()
        yx_model[tr_nr] = (y_model, x_model)
        
with open(f'yx_model_DMI-HYCOM_U.pkl', 'wb') as f:
    pickle.dump(yx_model,f)

In [ ]:
# Calculate yx_model in parallel
yx_model = {}
with ThreadPoolExecutor() as executor:
    futures = {executor.submit(calculate_tr_nr_yx_model, example_model_ds, key_lon_T, key_lat_T, key_x, key_y, dataset, tr_nr): tr_nr for tr_nr in tr_nr_values}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Calculating yx_model"):
        tr_nr, y_model, x_model = future.result()
        yx_model[tr_nr] = (y_model, x_model)
        
with open(f'yx_model_DMI-HYCOM_T.pkl', 'wb') as f:
    pickle.dump(yx_model,f)

In [ ]:
with open(f'yx_model_DMI-HYCOM_U.pkl', 'rb') as f:
    yx_model_U = pickle.load(f)

In [ ]:
with open(f'yx_model_DMI-HYCOM_T.pkl', 'rb') as f:
    yx_model_T = pickle.load(f)

In [ ]:
example_dmi_hycom

### Fill in overarching dataset with forecast variables

In [ ]:
# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

def process_forecasts(file_date, output_folder, model_name, key_lon, key_lat, dataset, yx_model):
    file_date_str = file_date.strftime('%Y%m%d')
    logging.info(f"Processing forecasts for date: {file_date_str}")

    # Try opening forecasts
    file_name_fc = os.path.join(output_folder, f"{model_name}_{file_date_str}.nc")

    alpha_rad = alpha_ds_U['__xarray_dataarray_variable__'] * np.pi / 180
    alpha_rad = alpha_rad.isel(ni=slice(xi_dmi_hycom_min,xi_dmi_hycom_max+1), nj=slice(yi_dmi_hycom_min,yi_dmi_hycom_max+1))

    # Try opening forecasts
    try:
        ds_fc = xr.open_dataset(file_name_fc)
        ds_fc['Uice'] = ds_fc['sea_ice_x_velocity'] * np.cos(alpha_rad) - ds_fc['sea_ice_y_velocity'] * np.sin(alpha_rad)
        ds_fc['Vice'] = ds_fc['sea_ice_x_velocity'] * np.sin(alpha_rad) + ds_fc['sea_ice_y_velocity'] * np.cos(alpha_rad)

        updated_dataset = dataset.copy()
        time_ds = dataset.time_ds

        for tr_nr, (y_model, x_model) in yx_model_U.items():
            mask_nan = ~(np.isnan(x_model) + np.isnan(y_model))
            for i in range(0, 240, 24):
                lt_int = f']{i}_{i+24}]'
                left_lt = ((ds_fc.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))> i+0.1
                right_lt = ((ds_fc.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))<= i+24+0.1
                sel_lt = np.where(left_lt*right_lt)[0]
                ds_fc_lt_int = ds_fc.isel(time=sel_lt)
                av_times_fc = ds_fc_lt_int['time'].values
                mask_av = pd.to_datetime(time_ds).isin(pd.to_datetime(av_times_fc))

                if np.sum(mask_av * mask_nan) > 0:
                    for var in ['Uice', 'Vice']:
                        #print('X')
                        #print(x_model[mask_av * mask_nan].astype(int))
                        #print('Y')
                        #print(y_model[mask_av * mask_nan].astype(int))
                        var_model_lt_int_path = ds_fc[var].sel({'time': time_ds[mask_av * mask_nan], key_x: x_model[mask_av * mask_nan].astype(int) - xi_dmi_hycom_min, key_y: y_model[mask_av * mask_nan].astype(int)-yi_dmi_hycom_min})
                        updated_dataset[var].loc[dict(model=model_name, tr_nr=tr_nr, lt_int=lt_int, time_ds=time_ds[mask_av * mask_nan])] = var_model_lt_int_path.values

        for tr_nr, (y_model, x_model) in yx_model_T.items():
            mask_nan = ~(np.isnan(x_model) + np.isnan(y_model))
            for i in range(0, 240, 24):
                lt_int = f']{i}_{i+24}]'
                left_lt = ((ds_fc.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))> i+0.1
                right_lt = ((ds_fc.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))<= i+24+0.1
                sel_lt = np.where(left_lt*right_lt)[0]
                ds_fc_lt_int = ds_fc.isel(time=sel_lt)
                av_times_fc = ds_fc_lt_int['time'].values
                mask_av = pd.to_datetime(time_ds).isin(pd.to_datetime(av_times_fc))

                if np.sum(mask_av * mask_nan) > 0:
                    for var in ds_fc.data_vars:
                        if var not in ['sea_ice_x_velocity', 'sea_ice_y_velocity', 'Uice', 'Vice']:
                            var_model_lt_int_path = ds_fc[var].sel({'time': time_ds[mask_av * mask_nan], key_x: x_model[mask_av * mask_nan].astype(int) - xi_dmi_hycom_min, key_y: y_model[mask_av * mask_nan].astype(int)-yi_dmi_hycom_min})
                            updated_dataset[var].loc[dict(model=model_name, tr_nr=tr_nr, lt_int=lt_int, time_ds=time_ds[mask_av * mask_nan])] = var_model_lt_int_path.values

        return updated_dataset

    except FileNotFoundError:
        logging.warning(f"File not found: {file_name_fc}")
        return dataset
    else:
        return dataset
        

    
# Process forecasts in parallel
with ThreadPoolExecutor() as executor:
    futures = [
        executor.submit(partial(process_forecasts, file_date, output_folder, model_name, key_lon, key_lat, dataset, yx_model))
        for file_date in tqdm(file_dates, desc="Processing forecasts")
    ]
    for future in tqdm(futures, desc="Updating dataset with forecast results"):
        result = future.result()
        if result is not None:
            dataset.update(result)


### Fill in overarching dataset with simulated drift trajectories

In [ ]:
def process_trajectories(file_date, output_subfolder, model_name, dataset):
    file_date_str = file_date.strftime('%Y%m%d')
    logging.info(f"Processing trajectories for date: {file_date_str}")

    file_name_traj = os.path.join(output_subfolder, f"trajs_{model_name}_{file_date_str}.nc")
    try:
        ds_traj = xr.open_dataset(file_name_traj)
    
        updated_dataset = dataset.copy()
        trajectory_sim = 0

        for tr_nr in np.unique(ds_traj['origin_marker'].values):
            if not np.isnan(tr_nr):
                trajectory_sim += 1
                for i in range(0, 240, 24):
                    lt_int = f']{i}_{i + 24}]'
                    left_lt = ((ds_traj.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))> i+0.1
                    right_lt = ((ds_traj.time.values - np.datetime64(file_date))/ np.timedelta64(1, 'h'))<= i+24+0.1
                    sel_lt = np.where(left_lt*right_lt)[0]                
                    time_traj = ds_traj.sel(trajectory=trajectory_sim)['time'].isel(time=sel_lt)
                    time_traj = time_traj.sel(time = (time_traj.time <= np.datetime64('2024-06-01T00:00')))
                    if len(time_traj.values)>0:
                        updated_dataset['lon_sim'].loc[dict(model=model_name, tr_nr=tr_nr, lt_int=lt_int, time_ds=time_traj)] = ds_traj['lon'].sel(trajectory=trajectory_sim, time=time_traj)
                        updated_dataset['lat_sim'].loc[dict(model=model_name, tr_nr=tr_nr, lt_int=lt_int, time_ds=time_traj)] = ds_traj['lat'].sel(trajectory=trajectory_sim, time=time_traj)

        return updated_dataset
        
    except FileNotFoundError:
        logging.warning(f"File not found: {file_name_traj}")
        return dataset
    
    

    
# Process trajectories in parallel
with ThreadPoolExecutor() as executor:
    futures = [
        executor.submit(partial(process_trajectories, file_date, output_subfolder, model_name, dataset))
        for file_date in tqdm(file_dates, desc="Processing trajectories")
    ]
    for future in tqdm(futures, desc="Updating dataset with trajectory results"):
        result = future.result()
        if result is not None:
            dataset.update(result)


### Plot to check data

In [ ]:
buoy_nr = 3
buoy_data = dataset.sel(tr_nr = buoy_nr)

colors = {
    'NERSC-NextSIM': 'tab:blue',
    'ECCC-RIOPS': 'tab:orange',
    'ECCC-RIOPScoupled': 'tab:green',
    'NRL-GOFS': 'tab:red',
    'MET-TOPAZ5': 'tab:purple',
    'MET-ROMS_m00': 'tab:cyan',
    'MET-ROMS_m01': 'tab:pink',
    'DMI-HYCOM':'tab:brown'
}

#['NERSC-NextSIM', 'ECCC-RIOPS', 'ECCC-RIOPScoupled', 'NRL-GOFS','MET-TOPAZ5', 'MET-ROMS_m00', 'MET-ROMS_m01']
#models_sel = []
models_sel = ['DMI-HYCOM']
target_date = '2024-04-16'  # Example date in YYYY-MM-DD format

lon_obs_b, lat_obs_b = buoy_data['lon_obs'], buoy_data['lat_obs']
plt.figure()
start = pd.to_datetime(target_date) + pd.DateOffset(seconds=1)
end = start + pd.DateOffset(days=5)
        
lon_obs_b.sel(time_ds=slice(start,end)).plot(color='k')
for model_name in models_sel:
    color = colors.get(model_name, 'black')
    
    # Plot dummy line for legend
    plt.plot([], [], color=color, label=model_name)
    
    for k in range(0,5):
        lt_int_sel = dataset['lt_int'].values[k]
        start = pd.to_datetime(target_date) + pd.DateOffset(seconds=1) + pd.DateOffset(days=k) 
        end = start + pd.DateOffset(days=1)
        lon_sim_lt_k = buoy_data['lon_sim'].sel(model=model_name, time_ds=slice(start,end), lt_int=lt_int_sel)
        lat_sim_lt_k = buoy_data['lat_sim'].sel(model=model_name, time_ds=slice(start,end), lt_int=lt_int_sel)
        lon_sim_lt_k.plot(color=color)

plt.legend(loc='upper left')
plt.grid(True)

In [ ]:
tr_nr = 2
dataset['lat_obs'].sel(tr_nr=tr_nr).plot()
dataset['lat_sim'].sel(tr_nr=tr_nr, model='DMI-HYCOM', lt_int=']0_24]').dropna(dim='time_ds', how='any').plot()

In [ ]:
dataset['SIC'].sel(tr_nr=4, model=model_name, lt_int=']0_24]').dropna(dim='time_ds', how='any').plot()
dataset['SIC'].sel(tr_nr=4, model='ECCC-RIOPS', lt_int=']0_24]').dropna(dim='time_ds', how='any').plot()
dataset['SIC'].sel(tr_nr=4, model='NRL-GOFS', lt_int=']0_24]').dropna(dim='time_ds', how='any').plot()

In [ ]:
dataset.to_netcdf('traj_ds_DMI-HYCOM.nc')

## Add satellite data

In [ ]:
# AMSR2 SIC
amsr2_ds = xr.open_dataset("sat_amj.nc")
# MODIS SIST
# same model grid as AROME Arctic
modis_ir_ds = xr.open_dataset(f"/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/observations/remotesensing/aa-modis-2024.nc",engine="netcdf4")
modis_ir_ds

In [ ]:
amsr2_ds

In [ ]:
with open(f'yx_AMSR2.pkl', 'rb') as f:
    yx_model = pickle.load(f)
time_ds = dataset.time_ds
for tr_nr, (y_model, x_model) in yx_model.items():
    mask_nan = ~(np.isnan(x_model) + np.isnan(y_model))
    # interpolate AMSR2 satellite ice concentration in time and space to position and time of temperature measurements
    dataset['AMSR2_SIC'].loc[dict(tr_nr=tr_nr, time_ds = time_ds[mask_nan])] = amsr2_ds['ice_conc'].interp(yc=y_model[mask_nan], xc=x_model[mask_nan], time=time_ds[mask_nan])

    



In [ ]:
modis_ir_ds

In [ ]:
# MODIS data uses same grid as AROMEArctic
example_model_ds = xr.open_dataset(f"/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/MET_AROMEArctic/AROME_ARCTIC/2024/04/15/arome_arctic_det_2_5km_20240415T00Z.nc", engine="netcdf4")
# add structure to dataset
modis_ir_ds = modis_ir_ds.assign_coords(
    y=('y', example_model_ds['y'].values),
    x=('x', example_model_ds['x'].values)
)
modis_ir_ds

In [ ]:
with open(f'yx_MODIS.pkl', 'rb') as f:
    yx_model = pickle.load(f)
time_ds = dataset.time_ds
for tr_nr, (y_model, x_model) in yx_model.items():
    mask_nan = ~(np.isnan(x_model) + np.isnan(y_model))
    # interpolate MODIS sea ice surface temperature in time and space to position and time of temperature measurements
    dataset['MODIS_Ts'].loc[dict(tr_nr=tr_nr, time_ds = time_ds[mask_nan])] = modis_ir_ds['modis_sist'].interp(y=y_model[mask_nan], x=x_model[mask_nan], time=time_ds[mask_nan])

    



In [ ]:
dataset['MODIS_Ts'].sel(tr_nr=4).dropna(dim='time_ds', how='any').plot()